## Problem Statement

Name: Jonathan Brosnan

In [ ]:
#!jupyter nbconvert --to html /content/Full_Code_Brosnan_NLP_RAG.ipynb

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install numpy==1.24.4
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==2.2.2
!pip install --upgrade numpy sentence-transformers langchain-community
#==5.1.1

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
import torch
import textwrap

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Variable and Function Definition Section

#### Variables

### The following variables are defined in this section and used throughout the project:
**Constants**:
 1. FLAG
 2. MODEL_DIR_PATH
 3. MODEL_NAME
 4. DATA_PATH
 5. EMBEDDED_MODEL_NAME
 6. OUT_DIR

**User Queries & LLM intended Prompts**:
 1. User Queries(5)
 2. Five Varying Prompt Templates(5)
 3. System Prompt Type

 **System & User Templates**:
 1. qna_system_message
 2. qna_user_message_template

 **Output Evaluation Variables**:
 1. Groundedness_rater_system_message
 2. Relevance_rater_system_message
 3. User_message_template



In [4]:
###**PROJECT CONSTANTS**

#1. Flag for toggling k_docs params in varied_func_and_params function
FLAG = True

#2. Path to the directory containing the Llama 2 model
MODEL_DIR_PATH = "TheBloke/Llama-2-13B-chat-GGUF"

#3. Name of the Llama 2 model file
MODEL_NAME = "llama-2-13b-chat.Q6_K.gguf"

#4. Path to the medical diagnosis manual pdf file
DATA_PATH = '/content/medical_diagnosis_manual.pdf'

#5. Name of the embedding model to be used for vectorization
EMBEDDED_MODEL_NAME = "all-MiniLM-L6-v2"
#model_name="all-MiniLM-L6-v2"
#6. Output directory to store or load the vector database
OUT_DIR = 'medical_db'

In [5]:
###**Lists of User Queries & LLM intended Prompts**

#1. User Queries(5)
user_query1 = "What is the protocol for managing sepsis in a critical care unit?"
user_query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
user_query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
user_query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
user_query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Used for the ending of each prompt.
promptend = " Do not refer to yourself as anything before providing an answer. Just provide the asked for information without any reference to yourself. Convert information formatted with bullets into a numbered list. Provide the answer after the heading ':' in paragraph form."
#2. Five varying prompt templates for the LLM to use(5)
prompt1 = "You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and accurate answer then respond with “Not enough data for an acceptable answer.” " + promptend
prompt2 = "You are a reliable medical assistant who provides evidence-based, medically accurate, evidence-based information responses just as a real medical assistant would. If the information provided does not support a clear and accurate conclusion, return “Not enough data for an acceptable answer.” In cases where the data is insufficient to ensure accuracy, respond with “Not enough data for an acceptable answer.” " + promptend
prompt3 = "Serve as a medical assistant with expertise in evidence-based medicine, providing accurate responses and transparently indicating any limitations in knowledge. If the available information is insufficient to provide an accurate response, reply with “Not enough data for an acceptable answer.” " + promptend
prompt4 = "You are a helpful and knowledgeable medical assistant who gives accurate, evidence-based health information and is honest when an answer is not known. When there isn’t adequate information to produce an honest and reliable answer, respond with “Not enough data for an acceptable answer.” " + promptend
prompt5 = "Act as a reliable medical assistant, offering evidence-based answers and acknowledging when information is unknown. In cases where the data is insufficient to ensure accuracy, respond with “Not enough data for an acceptable answer.” " + promptend

#3. System Prompt Type
sys_promp = "LLM Prompt Engineering Prompt:"

In [6]:
###**System & User Templates**

qna_promptend = " Answer using only the information explicitly stated in the context. Do not add assumptions, external knowledge, or interpretations. Do not refer to yourself as anything before providing an answer. Just provide the asked for information without any reference to yourself. If the answer provided includes bullets or a list of some kind, convert their format into a numbered list."


#1. System message for medical question-answering: instructs the LLM to answer solely based on the Merck Manual, maintaining accuracy and not referencing the source explicitly.
qna_system_message = """
You are a knowledgeable and helpful medical assistant. Provide accurate, concise, and medically reliable answers based solely on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use only the information available from this source to generate responses. If the available information is insufficient to support a clear and accurate answer,
state that you lack sufficient information to answer. Do not explicitly state sources such as Merck Manual, the context, or any external sources when answering questions."
"""


#2. Template for the user message: formats the retrieval-augmented context and user's question for the LLM, instructing it to answer strictly according to the given text.
qna_user_message_template = """Context: {context}

Question: {question}
"""

In [7]:
###**Output Evaluation Variables**

#1. System prompt for evaluating the groundedness of responses
groundedness_rater_system_message = """You are a groundedness evaluator for a retrieval-augmented generation (RAG) medical QA system.

You will be given:
- ###Question: the user query
- ###Context: retrieved passages used by the LLM system
- ###Answer: the model-generated response

Your job is to score how well the ###Answer is supported by the retrieved ###Context. Check whether each factual claim in the answer is explicitly stated or clearly supported by the context, and flag any hallucinations, unsupported additions, or contradictions.

Assign a groundedness score from 1 to 5:
1 - Not grounded: contradicts the context or contains mostly unsupported claims.
2 - Weakly grounded: only small portions are supported by the context.
3 - Moderately grounded: the main points are supported, but some claims are missing support.
4 - Mostly grounded: nearly all claims are supported with minor unsupported details.
5 - Fully grounded: every factual claim is directly supported by the context.

Return only the numeric rating (1-5).
"""

#2. System prompt for evaluating the relevance of responses to the user's query
relevance_rater_system_message = """You are a relevance evaluator for a retrieval-augmented generation (RAG) medical QA system.

You will be given:
- ###Question: the user query
- ###Context: retrieved passages used by the LLM system
- ###Answer: the model-generated response

Your job is to score how well the ###Answer addresses the ###Question. Evaluate whether the response directly resolves the user’s intent and covers the key requested details.

Assign a relevance score from 1 to 5:
1 - Completely irrelevant: does not answer the question.
2 - Weak relevance: slightly related but misses most key points.
3 - Partial relevance: answers some parts but omits important details.
4 - High relevance: addresses most key aspects with minor gaps.
5 - Fully relevant: directly and completely answers the question.

Return only the numeric rating (1-5).
"""


#3. Template for formatting user messages for evaluation, including the question, answer, rating, and context.
user_message_template = """
###Question: {question}
###Response: {answer}
###Rating(1-5):
###Context: {context}
"""

#### Functions

### The following functions are defined in this section and used throughout the project.
**Functions**:
 1. get_llm_instance
 2. wrap_text
 3. response
 4. varied_func_and_params
 5. varied_prompts_func_and_params
 6. get_chunk_content


In [ ]:
#1. Creates and returns an Llama LLM instance with appropriate parameters depending on whether a GPU is available.
def get_llm_instance(
    model_path  #model_path (str): Path to the pretrained Llama model checkpoint.
):
    # Try to detect if GPU is available (CUDA or ROCm)
    gpu_available = torch.cuda.is_available() or hasattr(torch, "has_mps") and torch.has_mps
    if gpu_available:
        # Use GPU-optimized parameters
        print("GPU MODEL USED")
        return Llama(
            model_path=model_path,
            n_ctx=2300,
            n_gpu_layers=38,
            n_batch=512
        )
    else:
        # Use CPU-optimized parameters
        print("CPU MODEL USED")
        return Llama(
            model_path=model_path,
            n_ctx=1024,
            n_cores=-2
        )

#2. Wraps the text at 100 characters for readability.
def wrap_text(text):
    # If the input is a dict, convert to string representation
    if isinstance(text, dict):
        text = str(text)
    # Replace newlines with spaces and clean up
    clean_text = text.replace('\n', ' ').strip()
    # Use textwrap to wrap at 100 chars, do not cut words
    return '\n'.join(textwrap.wrap(clean_text, width=100, break_long_words=False))

#3. Generates a response from the LLM using the given query and generation parameters.
def response(
    query,              # str: The input prompt or query for the LLM
    param_name,         # str: The parameter name being used
    max_tokens=128,     # int: Maximum number of tokens to generate in the response
    temperature=0,      # float: Sampling temperature (0=deterministic, >0 increases randomness)
    top_p=0.95,         # float: Top-p (nucleus) sampling probability threshold
    top_k=50            # int: Top-k sampling, sets maximum number of tokens to sample from at each step
):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    output_text_raw = model_output['choices'][0]['text']
    # Truncate if text exceeds 1000 items
    if len(output_text_raw) > 1000:
        output_text = output_text_raw[:1000] + '... '
    else:
        output_text = output_text_raw

    wrapped_output = wrap_text(output_text)
    print("\n" + param_name + wrapped_output)
    return output_text



#4. Template for formatting user messages for evaluation, including the question, answer, rating, and context.
def varied_func_and_params(
    response_func,  # function: The function used to generate responses (LLM or RAG)
    user_query,     # str: The user query or prompt to generate a response for
    rag=False       # bool: Whether to use RAG parameters (True) or standard LLM (False)
):
    # If rag is False, use standard LLM generation parameters for response_func with varying param values not including k_docs.
    if rag==False:
        print("1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50")
        #print(response_func(user_query), end = "\n")
        response_func(user_query, param_name="Response #1:")

        print("\n====================================================================================================")
        print("\n2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1")
        response_func(user_query, "\nResponse #2:", max_tokens=175, temperature=0.4, top_p=0.5, top_k=1)

        print("\n====================================================================================================")
        print("\n3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100")
        response_func(user_query, "\nResponse #3:", max_tokens=250, temperature=0.7, top_p=0.0, top_k=100)

        print("\n====================================================================================================")
        print("\n4. Custom Parameters: max_tokens=200, temperature=0.1, top_p=0.3, top_k=25")
        response_func(user_query, "\nResponse #4:", max_tokens=200, temperature=0.1, top_p=0.3, top_k=25)

        print("\n====================================================================================================")
        print("\n5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75")
        response_func(user_query, "\nResponse #5:", max_tokens=300, temperature=1.0, top_p=0.7, top_k=75)
    # Else, when rag is True, use generation params including k_docs
    else:
        print("1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50")
        #print(response_func(user_query), end = "\n")
        response_func(user_query)

        print("\n====================================================================================================")
        print("\n2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1")
        response_func(user_query, k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1)

        print("\n====================================================================================================")
        print("\n3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100")
        response_func(user_query, k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100)

        print("\n====================================================================================================")
        print("\n4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25")
        response_func(user_query, k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25)

        print("\n====================================================================================================")
        print("\n5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75")
        response_func(user_query, k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75)


#5. Tries multiple prompt templates, calling the given response function and parameter-variation function on each.
def varied_prompts_func_and_params(
    prompt_type,  # str: A string or instruction to prepend/describe the prompt type/context
    resp_func,    # function: The response function to be used for generating answers
    user_query    # str: The actual user question/query to be answered
):
    for i, prompt in enumerate([prompt1, prompt2, prompt3, prompt4, prompt5], 1):
        preview_length = 175  # Number of characters to show from the prompt
        print("\n====================================================================================================")
        print("====================================================================================================")
        print(f"\nUsing Prompt #{i}: {prompt[:preview_length]}{'...' if len(prompt) > preview_length else ''}")
        print("\n====================================================================================================")
        varied_func_and_params(resp_func, prompt_type + " " + prompt + "\n" + user_query)




#6. Get the embeddings for the first two chunks, handling both objects with 'page_content' attribute and plain text
def get_chunk_content(
    chunk #chunk: The document chunk, which can either have a 'page_content' attribute or be a plain string.
):
    return getattr(chunk, "page_content", chunk) if hasattr(chunk, "page_content") else str(chunk)


# Question Answering using LLM

#### Downloading and Loading the model

In [9]:
# Download the model file from the Hugging Face hub using the specified repo and filename
model_path = hf_hub_download(
    repo_id = MODEL_DIR_PATH,  # The repo to download from
    filename = MODEL_NAME      # The specific model file to fetch
)

# Initialize the LLM instance using the downloaded model file. Will adapt based on whether the runtime is using a GPU or a CPU.
llm = get_llm_instance(model_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


llama-2-13b-chat.Q6_K.gguf:   0%|          | 0.00/10.7G [00:00<?, ?B/s]

GPU MODEL USED


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## Parameter Description using vanilla LLM functions

**Output Description:** Each query is tested with 5 different parameter combinations. That means each query produces 5 total responses so there will be a total of 25 responses in the is section(5 x 5)

*Note: Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.*

Describing how the model outputs might be effected by parameter values of each combination when using: **varied_func_and_params(response, user_query#)** and **response(user_query#, max_tokens, temperature, top_p, top_k)**

---
1) **Default Parameters:** max_tokens=128, temperature=0, top_p=0.95, top_k=50
  
    **Output Indications:** These parameters produce short, highly deterministic responses which may lack creativity or variation. A temperature of 0 means no randomness (the model gives the most likely next token at every step), top_p=0.95 allows common, high-probability words, and top_k=50 gives the model a moderate pool of possible words to choose from. With max_tokens=128 comes a lack of extremely creative or thorough responses, but it makes up for it by providing focused, concise responses.
---
2) **Custom Parameters:** max_tokens=175, temperature=0.4, top_p=0.5, top_k=1

    **Output Indications:** Expect answers with only modest variability, still very focused and concise. This configuration slightly increases randomness with a temperature of 0.4, but using top_k=1 means the model will always pick the single most likely token (even though top_p is lower, it is redundant when top_k=1). The higher max_tokens value of 175 allows for a longer answer than the default parameter max_token value of 128.
---
3) **Custom Parameters:** max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

    **Output Indications:** A higher temperature (0.7) increases randomness and diversity in the model's responses. The top_p=0.0 value effectively disables nucleus sampling, so only top_k=100 is used, which gives the model many token choices. The max_token value of 250 allows for a long output window, so responses will be more creative and varied, but might sometimes lose focus or coherence.
---
4) **Custom Parameters:** max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

    **Output Indications:** Use these parameters for reliability, accuracy and relevance rather than creativity. The max_tokens=200 value gives moderate response length. The low temperature (0.1) value yields mostly deterministic outputs, but the top_p=0.3 value restricts the candidate pool to only the most probable tokens, usually resulting in concise, safe answers. The top_k=25 value moderately increases potential variability within the top tokens.
---
5) **Custom Parameters:** max_tokens=300, temperature=1.0, top_p=0.7, top_k=75

    **Output Indications:** With the highest temperature (1.0), outputs are at their most diverse and creative, though possibly less reliable. The top_p=0.7 value and the top_k=75 value both limit the response space, encouraging some focus but still allowing a lot of variation. The largest max_tokens value of 300 may result in the most thorough or elaborate answers, excellent for brainstorming or open-ended questions


    

## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [19]:
#Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.
varied_func_and_params(response, user_query1)

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause
of death in hospitals. The management of sepsis in a critical care unit requires a systematic
approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a
general protocol for managing sepsis in a critical care unit:  1. Early recognition:     * Use a
standardized definition of sepsis, such as the Sepsis-3 definition (Systemic Inflammatory Response
Syndrome with organ dysfunction).


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause
of death in hospitals. The management of sepsis in a critical care unit requires a systematic
approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a
general protocol for managing sepsis in a critical care unit:  1. Early recognition:     * Use a
standardized definition of sepsis, such as the Sepsis-3 definition (Systemic Inflammatory Response
Syndrome with organ dysfunction).    * Screen all patients for signs of sepsis, including fever,
tachycardia, tachypnea, and altered mental status.  * Use a sepsis screening tool, such as


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause
of death in hospitals. The management of sepsis in a critical care unit requires a systematic
approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a
general protocol for managing sepsis in a critical care unit:  1. Early recognition:     * Use a
standardized definition of sepsis, such as the Sepsis-3 definition (Systemic Inflammatory Response
Syndrome with organ dysfunction).    * Screen all patients for signs of sepsis, including fever,
tachycardia, tachypnea, and altered mental status.  * Use a sepsis screening tool, such as the Quick
Sequential Organ Failure Assessment (qSOFA) or the Systemic Inflammatory Response Syndrome (SIRS)
criteria. 2. Initial assessment:     * Evaluate the patient's vital signs, including temperature,
blood pressure, tachycardia, and tachypnea.


4. Custom Parameters: max_tokens=200, temperature=0.1, top_p=0.

Llama.generate: prefix-match hit




Response #4:Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause
of death in hospitals. The management of sepsis in a critical care unit requires a systematic
approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a
general protocol for managing sepsis in a critical care unit:  1. Early recognition:     * Use a
standardized definition of sepsis, such as the Sepsis-3 definition (Sepsis-3 Task Force, 2016).
* Screen all patients for signs of sepsis, including fever, tachycardia, tachypnea, and altered
mental status.  * Use a severity score, such as the Sequential Organ Failure Assessment (SOFA) score
(Vincent et al., 1996), to evaluate the


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:What are the signs and symptoms of sepsis? What are the criteria for diagnosing sepsis? What are the
initial management strategies for sepsis? What are the goals of treatment for sepsis? What is the
role of antibiotics in the management of sepsis? What other interventions may be used in addition to
antibiotics to manage sepsis? What are the potential complications of sepsis, and how can they be
managed? What is the prognosis for patients with sepsis, and what factors influence this prognosis?
Sepsis is a life-threatening condition that arises from an immune response to an infection. It is a
leading cause of morbidity and mortality in critical care units worldwide. The management of sepsis
involves early recognition, prompt treatment, and ongoing monitoring of the patient's clinical
status. The protocol for managing sepsis in a critical care unit typically includes the following
steps: 1. Early recognition and diagnosis: Patients with suspected sepsis should be evaluated
i

### ✅ Best overall: **Response #3**

**Why it’s the best:**

1. Most complete response so far. It actually starts laying out a **step-by-step** protocol.
2. More clinically aligned via mentions **qSOFA / SIRS** and begins a structured workflow.
3. Feels very professional by using easily readible lists of medical protocols. It reads like something you’d expect from a clinical checklist, not just a general description.


---

## ❌ Worst overall: **Response #1**

**Why it’s the worst:**

1. **Too incomplete:** It barely starts the protocol and doesn’t provide actionable steps.
2. Contains a serious medical accuracy issue:
	* It describes Sepsis-3 as **“SIRS with organ dysfunction”**, which is **not correct**.
	* Sepsis-3 moved away from SIRS-based definitions.
3. **Not useful to the patient:** It reads like an intro paragraph + a broken bullet point.

## Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [20]:
varied_func_and_params(response, user_query2)

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:Answer:  The most common symptoms of appendicitis include:  1. Severe pain in the abdomen, usually
starting near the belly button and then moving to the lower right side of the abdomen. 2. Nausea and
vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and guarding (muscle tension). 6.
Abdominal swelling.  If you suspect that you or someone else may have appendicitis, it is important
to seek medical attention immediately


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:Answer:  Symptoms of Appendicitis:  The most common symptoms of appendicitis include:  1. Severe
pain in the abdomen, usually starting near the belly button and then moving to the lower right side
of the abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
guarding (muscle tension). 6. Abdominal swelling. 7. Diarrhea or constipation.  If you experience
any of these symptoms, it is essential to seek medical attention immediately, as appendicitis can be
a life-threatening condition if left untreated.  Can Appendicitis Be Cured


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:Answer:  Symptoms of Appendicitis:  The most common symptoms of appendicitis include:  1. Severe
pain in the abdomen, usually starting near the belly button and then moving to the lower right side
of the abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
guarding (muscle tension). 6. Abdominal swelling. 7. Diarrhea or constipation.  If you experience
any of these symptoms, it is essential to seek medical attention immediately, as appendicitis can be
a life-threatening condition if left untreated.  Can Appendicitis Be Cured via Medicine?  In some
cases, appendicitis may be treated with antibiotics and other supportive care, but surgery is
usually necessary to remove the inflamed appendix. Antibiotics can help reduce the infection and
alleviate symptoms, but they cannot cure the condition itself.  Surgical Procedure for Append


4. Custom Parameters: max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit




Response #4:Answer:  Symptoms of Appendicitis:  The most common symptoms of appendicitis include:  1. Severe
pain in the abdomen, usually starting near the belly button and then moving to the lower right side
of the abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
guarding (muscle tension). 6. Abdominal swelling. 7. Diarrhea or constipation.  If you experience
any of these symptoms, it is essential to seek medical attention immediately, as appendicitis can be
a life-threatening condition if left untreated.  Can Appendicitis Be Cured via Medicine?  In some
cases, appendicitis may be treated with antibiotics and other supportive care


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:Answer: Appendicitis is a medical emergency that requires prompt treatment. The common symptoms of
appendicitis include:  1. Severe pain in the abdomen, usually starting near the navel and then
moving to the lower right side of the abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4.
Fever. 5. Abdominal tenderness and guarding (muscle tension). 6. Abdominal swelling. 7. Diarrhea or
constipation.   Appendicitis is caused by an inflammation of the appendix, a small pouch-like organ
attached to the large intestine. The exact cause of appendicitis is not known, but it may be due to
obstruction of the appendix by feces, foreign bodies, or tumors.  Medications can help relieve
symptoms such as pain and fever, but they cannot cure appendicitis. Antibiotics may be given to
prevent infection, but they are not effective in treating the inflamed appendix. Surgery is the only
way to treat appendicitis.  The surgical procedure for treating appendicitis is called an
appendectomy, w

### ✅ Best Response: **Response #5**

### Why it’s the best

1. **Actually answers all 3 parts** of the question:

	1. **Symptoms**
	2. **Can medicine cure it?** (says no)
	3. **What surgery is used?** (appendectomy)

2. **Patient-friendly and clear:** It calls appendicitis a **medical emergency**, which is an important safety cue.
3. Adds helpful context without going off track.
4. Briefly explains what appendicitis is and why it’s serious.



---

## ❌ Worst Response: **Response #1**

### Why it’s the worst

1. Doesn’t answer the actual question fully.

	* It lists symptoms ✅
	* But it **doesn’t address** whether it can be cured with medicine
	* And it **doesn’t name** the surgical procedure (appendectomy)
  
2. It’s basically a symptom list + “seek medical attention,” which is fine, but **not sufficient** given what the patient asked.



## Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [21]:
varied_func_and_params(response, user_query3)

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here
are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These injections can help suppress the immune system and promote hair growth. They are
usually administered every 4-6 weeks. 2. Topical corticosteroids: Over-the-counter or prescription
topical corticosteroids can be applied directly to the affected area to reduce inflammation and
promote hair growth. 3.


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here
are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These injections can help suppress the immune system and promote hair growth. They are
usually administered every 4-6 weeks. 2. Topical corticosteroids: Over-the-counter or prescription
topical corticosteroids can be applied directly to the affected area to reduce inflammation and
promote hair growth. 3. Minoxidil (Rogaine): This is a solution that is applied to the scalp to
stimulate hair growth and slow down hair loss. It is available over-the-counter. 4. Anthralin


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here
are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These injections can help suppress the immune system and promote hair growth. They are
usually administered every 4-6 weeks. 2. Topical corticosteroids: Over-the-counter or prescription
topical corticosteroids can be applied directly to the affected area to reduce inflammation and
promote hair growth. 3. Minoxidil (Rogaine): This is a solution that is applied to the scalp to
stimulate hair growth and slow down hair loss. It is available over-the-counter. 4. Anthralin
(Psoralen): This is a medication that is applied to the skin to reduce inflammation and promote hair
growth. It is usually used in combination with ultraviolet light therapy. 5. Phototherapy: Exposure
to specific wavelengths of light, such as ultraviolet B (UVB) or narrowband


4. Custom Parameters: max_tok

Llama.generate: prefix-match hit




Response #4:Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here
are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These injections can help suppress the immune system and promote hair growth. They are
usually administered every 4-6 weeks. 2. Topical corticosteroids: Over-the-counter or prescription
topical corticosteroids can be applied directly to the affected area to reduce inflammation and
promote hair growth. 3. Minoxidil (Rogaine): This is a solution that is applied to the scalp to
stimulate hair growth and slow down hair loss. It is available over-the-counter. 4. Anthralin
(Psoralen): This is a medication that is applied to the skin to reduce inflammation and promote hair


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:Answer: Sudden patchy hair loss, also known as alopecia, can have several causes. Some of the most
common causes include:  1. Hormonal Imbalance: An imbalance of hormones such as thyroid hormones,
androgen hormones, or estrogen hormones can cause hair loss. 2. Autoimmune Disorders: Conditions
like alopecia areata, lupus, and rheumatoid arthritis can cause patchy hair loss. 3. Infections:
Fungal infections such as ringworm, bacterial infections, and viral infections can lead to hair
loss. 4. Skin Conditions: Certain skin conditions like eczema, psoriasis, and seborrheic dermatitis
can cause hair loss. 5. Nutrient Deficiencies: Lack of essential nutrients such as iron, zinc, or
biotin can lead to hair loss. 6. Physical Stress: Physical stress caused by events like childbirth,
surgery, or a severe illness can cause hair loss. 7. Medical Conditions: Certain medical conditions
like polycystic ovary syndrome (PCOS), hypothyroidism, and hyperthyroidism can cause patchy hair
loss

### ✅ Best Response: **Response #5**

### Why it’s the best

1. Actually answers the *cause* part of the question well. It gives a broad, patient-friendly list of possible causes (autoimmune issues, infections like ringworm, thyroid problems, nutrient deficiencies, stress, etc.).
2. **Covers multiple plausible diagnoses**, which matters because “patchy bald spots” isn’t always alopecia areata.
3. **Safer overall** than the others because it doesn’t jump straight into treatments like injections without first considering causes that need different management (ex: fungal infection).


---

## ❌ Worst Response: **Response #1**

### Why it’s the worst

1. **Incomplete and cut off.** It stops at “3.” and never finishes.
2. It **assumes it’s alopecia areata immediately**, which isn’t always true. Patchy hair loss can also be:

	* fungal infection (tinea capitis)
	* traction alopecia
	* trichotillomania
	* scarring inflammatory conditions

	Jumping to one diagnosis without warning is not patient-safe.

3. It gives **treatment suggestions without enough context**, which can mislead patients (ex: steroid injections are medical procedures and not always appropriate).


## Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [22]:
varied_func_and_params(response, user_query4)

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:There are several treatments that may be recommended for a person who has sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the location and severity of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, inflammation, and anxiety. 2. Rehabilitation
therapy: To help regain lost function and improve cognitive, physical, and emotional abilities.


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:There are several treatments that may be recommended for a person who has sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the location and severity of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, inflammation, and anxiety. 2. Rehabilitation
therapy: To help regain lost function and improve cognitive, physical, and emotional abilities. This
may include physical therapy, occupational therapy, speech therapy, and cognitive therapy. 3.
Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:There are several treatments that may be recommended for a person who has sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the location and severity of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, inflammation, and anxiety. 2. Rehabilitation
therapy: To help regain lost function and improve cognitive, physical, and emotional abilities. This
may include physical therapy, occupational therapy, speech therapy, and cognitive therapy. 3.
Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged
tissue. 4. Neurostimulation therapies: Such as transcranial magnetic stimulation (TMS) or
transcranial direct current stimulation (tDCS), which can help improve cognitive function and reduce
symptoms of depression

Llama.generate: prefix-match hit




Response #4:There are several treatments that may be recommended for a person who has sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the location and severity of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, inflammation, and anxiety. 2. Rehabilitation
therapy: To help regain lost function and improve cognitive, physical, and emotional abilities. This
may include physical therapy, occupational therapy, speech therapy, and cognitive therapy. 3.
Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged
tissue. 4. Neurostimulation therapies: Such as transcranial magnetic stim


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:There are several treatment options available for individuals who have sustained a physical injury
to brain tissue, resulting in temporary or permanent impairment of brain function. The specific
treatment plan will depend on the nature and severity of the injury, as well as the individual's
overall health and medical history. Some common treatments for brain injuries include:  1.
Medications: Medications such as pain relievers, anti-seizure drugs, and antidepressants may be
prescribed to manage symptoms and reduce inflammation. 2. Rehabilitation therapy: Physical,
occupational, and speech therapy can help individuals regain lost function and improve their quality
of life. 3. Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or
repair damaged tissue. 4. Cognitive rehabilitation: This type of therapy can help individuals with
cognitive impairments, such as memory loss and difficulty with concentration, to improve their
cognitive function. 5. 

### ✅ Best Response: **Response #5**

### Why it’s the best

1. Has the most complete and patient-friendly information. It gives a clear list of the main treatment categories that actually reflect how brain injuries are managed:

	* **Medications** (including important examples like anti-seizure meds)
	* **Rehabilitation therapy** (PT/OT/speech)
	* **Surgery** when needed
	* **Cognitive rehab**
  
2. Better specificity without going off the rails. It avoids being overly vague while still staying general enough for a patient.


---

## ❌ Worst Response: **Response #1**

### Why it’s the worst

1. The treatment suggestions list is too short and incomplete. It only gives:

	* Medications
	* Rehab therapy

    This is not a sufficient number of treatments options to give a patient.

2. **Misses critical parts of treatment** that patients expect to hear about:

	* possible **surgery**
	* **seizure prevention**
	* monitoring for complications (swelling/bleeding)
	* longer-term recovery support
  
3. “Medications for pain, inflammation, anxiety” is also a bit vague and can mislead. Brain injury medications are often focused on things like seizures, agitation, sleep, nausea, etc., not just inflammation.

## Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [23]:
varied_func_and_params(response, user_query5)

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:A person who has fractured their leg during a hiking trip requires prompt medical attention to
ensure proper healing and prevent complications. Here are some necessary precautions and treatment
steps:  1. Stop all activities: The first step is to stop all activities, especially weight-bearing
ones, to avoid exacerbating the injury. This includes hiking, running, or any other physical
activity that may put pressure on the fractured leg. 2. Seek medical attention: It's essential to
seek medical attention as soon as possible, especially if the fracture is


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:A person who has fractured their leg during a hiking trip requires prompt medical attention to
ensure proper healing and prevent complications. Here are some necessary precautions and treatment
steps:  1. Stop all activities: The first step is to stop all activities, especially weight-bearing
ones, to avoid exacerbating the injury. This includes hiking, running, or any other physical
activity that may put pressure on the affected leg. 2. Seek medical attention: It's essential to
seek medical attention as soon as possible, especially if the fracture is displaced or if there are
signs of nerve or blood vessel damage. A healthcare professional will perform a physical examination
and order imaging tests, such as X-rays or CT scans, to determine the extent of the


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:A person who has fractured their leg during a hiking trip requires prompt medical attention to
ensure proper healing and prevent complications. Here are some necessary precautions and treatment
steps:  1. Stop all activities: The first step is to stop all activities, especially weight-bearing
ones, to avoid exacerbating the injury. This includes hiking, running, or any other physical
activity that may put pressure on the affected leg. 2. Seek medical attention: It's essential to
seek medical attention as soon as possible, especially if the fracture is displaced or if there are
signs of nerve or blood vessel damage. A healthcare professional will perform a physical examination
and order imaging tests, such as X-rays or CT scans, to determine the extent of the injury. 3.
Immobilize the leg: To prevent further damage and promote healing, the affected leg should be
immobilized using a splint, cast, or brace. This will help keep the bone in place and reduce pain.
4. Manage pai

Llama.generate: prefix-match hit




Response #4:A person who has fractured their leg during a hiking trip requires prompt medical attention to
ensure proper healing and prevent complications. Here are some necessary precautions and treatment
steps:  1. Stop all activities: The first step is to stop all activities, especially weight-bearing
ones, to avoid exacerbating the injury. This includes hiking, running, or any other physical
activity that may put pressure on the affected leg. 2. Seek medical attention: It's essential to
seek medical attention as soon as possible, especially if the fracture is displaced or if there are
signs of nerve or blood vessel damage. A healthcare professional will perform a physical examination
and order imaging tests, such as X-rays or CT scans, to determine the extent of the injury. 3.
Immobilize the leg: To prevent further damage and promote healing, the affected leg should


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:A person who has fractured their leg while hiking may require immediate medical attention to prevent
further complications. The following are some necessary precautions and treatment steps:  1. Stop
all activity and rest: The first step is to stop all activity and rest the affected leg. This will
help reduce the risk of further damage or complications.  2. Apply a splint: A splint can be applied
to immobilize the fractured leg and reduce pain. The splint should be applied in a way that
maintains the normal position of the bone and joint.  3. Elevate the affected area: Elevating the
fractured leg above the level of the heart can help reduce swelling and improve blood flow.  4.
Apply ice: Applying ice to the affected area can help reduce pain and swelling. However, it should
be done with caution to avoid frostbite.  5. Monitor for signs of infection: It is essential to
monitor the person for signs of infection, such as redness, swelling, or increased pain. If any of
these s

### ✅ Best Response: **Response #5**

### Why it’s the best

1. Most useful for the **during a hiking trip** setting. It gives practical field first-aid steps the person can actually do right away:

	* Stop activity/rest
	* Splint/immobilize
	* Elevate
	* Ice (**with breaks every 20 min**)
  
2. The response is clear and actionable. It reads like a real emergency checklist, which is what someone needs outdoors.
3. It includes monitoring for complications (infection signs), which shows it’s thinking about **follow-up risk**, not just pain.


---

## ❌ Worst Response: **Response #1**

### Why it’s the worst

1. Too incomplete and seems to cut off the response early.
2. It gives only:

	* stop activities
	* seek medical attention … but doesn’t provide any real how-to steps for the situation (splinting, keeping warm, avoiding movement, etc.)
  
3. For a patient stuck on a trail, the advice is too vague and obvious. Everyone knows that they should stop moving and go to a doctor when they are in that situation. Responses like this are not helpful.

## Observations
*Note: Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.*

---

###**Results:**


Best Responses: 3, **5, 5, 5, 5**

Worst Responses: **1, 1, 1, 1, 1**

---

###**Conclusions:**

Response 5 was the best response for 4 of the 5 queries while response 1 was the worst for all 5 queries. Response 5 was the best a majority of the time because it has the highest max_tokens parameter value so it has more data to create a response from. Response 1 has the lowest max_token value so that's why it was the worst response for all 5 of the queries.


# Question Answering using LLM with Prompt Engineering

#### Using multiple prompt templates in conjunction with the previously used LLM functions

**Output Description:** Each query is run through the 5 varying prompts that are competing against each other to find out which is the best prompt. For each prompt, there are 5 differing parameter combinations that are used by the LLM function to produce 5 responses per prompt.Therefore each query gets a total of 25 responses. That means the total number of responses for the entire section will be 125(25 x 5).

The parameter combinations are the same as the first section so they will have the same effect on the output of these prompts.

*Note: Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.*

In [ ]:
# These 5 varied prompts will be used to see how differing wording and sentence structure effects the model's responses.
for i, (name, prompt) in enumerate(
    [
        ("Prompt #1", prompt1),
        ("Prompt #2", prompt2),
        ("Prompt #3", prompt3),
        ("Prompt #4", prompt4),
        ("Prompt #5", prompt5)
    ], 1):
    print(f"{name}: {wrap_text(prompt)}\n")

Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based
healthcare responses. If there is not enough information to develop a sufficient and accurate answer
then respond with “Not enough data for an acceptable answer.”  Do not refer to yourself as anything
before providing an answer. Just provide the asked for information without any reference to
yourself. Convert information formatted with bullets into a numbered list. Provide the answer after
the heading ':' in paragraph form.

Prompt #2: You are a reliable medical assistant who provides evidence-based, medically accurate, evidence-based
information responses just as a real medical assistant would. If the information provided does not
support a clear and accurate conclusion, return “Not enough data for an acceptable answer.” In cases
where the data is insufficient to ensure accuracy, respond with “Not enough data for an acceptable
answer.”  Do not refer to yourself as anything before prov

## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [28]:
#Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.
varied_prompts_func_and_params(sys_promp, response, user_query1)



Using Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and...

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:: The management of sepsis in a critical care unit involves a multidisciplinary approach that
includes early recognition, prompt administration of antibiotics and supportive care, and careful
monitoring of vital signs and organ function. The following is a general protocol for managing
sepsis in a critical care unit:  1. Early recognition and activation of the sepsis team: Sepsis
should be suspected in any patient with a systemic inflammatory response syndrome (SIRS) and
evidence of organ dysfunction. The sepsis team, which includes intensivists, nurs


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:: The management of sepsis in a critical care unit involves a multidisciplinary approach that
includes early recognition, prompt administration of antibiotics and supportive care, and careful
monitoring of vital signs and organ function. The following is a general protocol for managing
sepsis in a critical care unit:  1. Early recognition and activation of the sepsis team: Sepsis
should be suspected in any patient with suspected or confirmed infection who is experiencing signs
of severe systemic inflammatory response syndrome (SIRS), such as fever, tachycardia, tachypnea, or
confusion. The sepsis team should be activated immediately, and the patient should be promptly
evaluated and treated. 2. Administration of antibiotics: Broad-spectrum antib


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:: The management of sepsis in a critical care unit involves a multidisciplinary approach that
includes early recognition, prompt administration of antibiotics and supportive care, and careful
monitoring of vital signs and organ function. Here are the key components of the protocol for
managing sepsis in a critical care unit:  1. Early recognition: Sepsis is a life-threatening
condition that requires early recognition and prompt intervention. Critical care nurses and other
healthcare professionals should be aware of the signs and symptoms of sepsis, which include fever,
tachycardia, tachypnea, confusion, and altered mentation. 2. Administration of antibiotics: The
administration of broad-spectrum antibiotics is a critical component of sepsis management. The
choice of antibiotic should be based on the suspected cause of sepsis, such as pneumonia or urinary
tract infection. 3. Supportive care: In addition to antibiotics, patients with sepsis require
supportive care to manage

Llama.generate: prefix-match hit




Response #4:: The management of sepsis in a critical care unit involves a multidisciplinary approach that
includes early recognition, prompt administration of antibiotics and supportive care, and careful
monitoring of vital signs and organ function. Here are the key components of the protocol for
managing sepsis in a critical care unit:  1. Early recognition: Sepsis is a life-threatening
condition that requires early recognition and prompt intervention. Critical care nurses and other
healthcare professionals should be aware of the signs and symptoms of sepsis, which include fever,
tachycardia, tachypnea, confusion, and altered mentation. 2. Administration of antibiotics: The
administration of broad-spectrum antibiotics is a critical component of sepsis management. The
choice of antibiotic should be based on the suspected cause of sepsis, such as p


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:The prompt requires you to act as a reliable medical assistant and provide evidence-based answers
while acknowledging when information is unknown. You should avoid referring to yourself before
providing an answer and instead use a neutral tone to present the information. Additionally, you
should convert any bulleted information into a numbered list for easier reading.  Here's the answer
to the prompt:  : Protocol for Managing Sepsis in a Critical Care Unit  Sepsis is a life-threatening
condition that can arise from an infection, and it is essential to have a well-established protocol
for managing it in a critical care unit. The Surviving Sepsis Campaign (SSC) guidelines provide
evidence-based recommendations for the management of sepsis. Here are the key components of the
protocol:  1. Early recognition and activation of the sepsis team: The protocol should emphasize the
importance of early recognition of sepsis, including signs of organ dysfunction, and prompt
activation

### ✅ Best Response

## **Best = Prompt #3 -> Response #5**

**Parameter combo:** `max_tokens=300, temperature=1.0, top_p=0.7, top_k=75`

### Why it's the best

1. This one gives the most complete and **protocol-style** answer that a patient could understand as a clear **step-by-step** process:

	* Recognizes sepsis early using common clinical signs
	* Antibiotics within 1 hour (timing is a key sepsis priority)
	* Fluid resuscitation included (core of early sepsis treatment)
	* Adds important ICU targets (**MAP ≥ 65**, urine output goal), which makes it feel like a real critical care protocol. Reads like an actual workflow: **identify → treat → stabilize → monitor**

---

# ❌ Worst Response

## **Worst = Prompt #1 -> Response #1**

**Parameter combo:** `max_tokens=128, temperature=0, top_p=0.95, top_k=50`

### Why it's the worst

This one is the least useful and least safe because it’s incomplete and ends mid-sentence:

1. Stops abruptly (“…intensivists, nurs”) — not usable as a protocol
2. Doesn’t reach the key protocol items (fluids, vasopressors, labs, cultures, monitoring, etc.)
3. Uses an outdated framing: **“SIRS + organ dysfunction”** isn’t how modern sepsis definitions are typically taught or applied now, so it risks confusion


## Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [29]:
varied_prompts_func_and_params(sys_promp, response, user_query2)



Using Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and...

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:: Appendicitis is a medical emergency that occurs when the appendix, a small pouch-like organ
located in the lower right abdomen, becomes inflamed and infected. The common symptoms of
appendicitis include:  1. Severe pain in the lower right abdomen that begins suddenly and worsens
over time. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
swelling. 6. Abdominal guarding, which is the muscle t


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:: Appendicitis is a medical emergency that occurs when the appendix, a small pouch-like organ
located in the lower right abdomen, becomes inflamed and infected. The common symptoms of
appendicitis include:  1. Severe pain in the lower right abdomen that begins suddenly and worsens
over time. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
swelling. 6. Abdominal guarding, which is the muscle tension in the abdomen to guard the area from
the pain. 7. Rigidity in the abdomen. 8. Palpable mass in the right lower quadrant of the abdomen.


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:: Appendicitis is a medical emergency that requires prompt treatment to prevent complications and
potentially life-threatening outcomes. The common symptoms of appendicitis include:  1. Severe pain
in the abdomen, usually starting near the navel and then moving to the lower right side of the
abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and guarding
(muscle tension). 6. Abdominal swelling.  While appendicitis can be treated with antibiotics in some
cases, surgery is usually necessary to remove the inflamed appendix. The surgical procedure most
commonly used to treat appendicitis is a laparoscopic appendectomy, where a small incision is made
in the abdomen and a laparoscope (a thin tube with a camera and light) is inserted to visualize the
appendix. The inflamed appendix is then removed through the small incision.  In some cases, an open
appendectomy may be necessary if the


4. Custom Parameters: max_tokens=200, temperature=0.1, t

Llama.generate: prefix-match hit




Response #4:: Appendicitis is a medical emergency that requires prompt treatment to prevent complications and
potentially life-threatening outcomes. The common symptoms of appendicitis include:  1. Severe pain
in the abdomen, usually starting near the navel and then moving to the lower right side of the
abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and guarding
(muscle tension). 6. Abdominal swelling.  While appendicitis can be treated with antibiotics in some
cases, surgery is usually necessary to remove the inflamed appendix. The surgical procedure most
commonly used to treat appendicitis is a laparoscopic appendectomy, where a small incision is made
in the abdomen and a laparoscope (


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:: Appendicitis is a medical emergency that requires prompt treatment to prevent complications and
potentially life-threatening outcomes. The common symptoms of appendicitis include:  1. Severe pain
in the abdomen, usually starting near the belly button and then moving to the lower right side of
the abdomen. 2. Nausea and vomiting. 3. Loss of appetite. 4. Fever. 5. Abdominal tenderness and
guarding (muscle tension). 6. Abdominal swelling.  While appendicitis can be treated with
antibiotics, surgery is usually required to remove the inflamed appendix. The surgical procedure
most commonly used to treat appendicitis is a laparoscopic appendectomy, where a small camera and
specialized instruments are inserted through small incisions in the abdomen to remove the inflamed
appendix. In some cases, an open appendectomy may be necessary if the appendix has ruptured or if
there are other complications present.  It is important to note that not all cases of appendicitis
can be cured 

### ✅ Best Response

## **Best = Prompt #2 -> Response #5**

**Parameter combo:** `max_tokens=300, temperature=1.0, top_p=0.7, top_k=75`

### Why it’s the best

1. Answers all parts of the question:

	* Lists **common symptoms**
	* Addresses **medicine vs cure** (mentions antibiotics may be used in mild cases)
	* Clearly states **surgery is usually needed**
	* Names the correct procedure: **appendectomy**
	* Mentions both **standard incision** and **laparoscopic approach**
  
2. Balanced and medically reasonable

	* Doesn’t oversimplify
	* The “mild cases may use antibiotics” wording adds realism and nuance
  
3. **Safety + urgency included:** Warns appendicitis can lead to serious complications if untreated


---

# ❌ Worst Response

## **Worst = Prompt #1 -> Response #1**

**Parameter combo:** `max_tokens=128, temperature=0, top_p=0.95, top_k=50`

### Why it’s the worst

1. **Incomplete / cut off mid-sentence:** '...which is the muscle t...'
2. Does NOT answer the full question:

	* It only covers symptoms
	* It never answers whether it can be cured by medicine
	* It never names the surgery (**appendectomy**)
  
3. Not usable for a patient because it stops before the most important treatment guidance



## Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [31]:
varied_prompts_func_and_params(sys_promp, response, user_query3)



Using Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and...

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:: There are several effective treatments and solutions for addressing sudden patchy hair loss,
commonly seen as localized bald spots on the scalp. The possible causes of this condition include:
1. Telogen effluvium: This is a common cause of sudden hair loss, where there is an increase in the
number of hair follicles that stop growing and enter the resting phase, leading to excessive
shedding. 2. Stress: Physical or emotional stress can cause hair loss by altering the hormone
balance in the body, leading to an increase in dihydrotest


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:: Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. The
most effective treatments and solutions for addressing this condition include:  1. Topical
corticosteroids: These medications can help reduce inflammation and promote hair growth. They are
applied directly to the affected area and can be purchased over-the-counter or prescribed by a
dermatologist. 2. Minoxidil: This medication is applied to the scalp and can help stimulate hair
growth and slow down hair loss. It is available over-the-counter and is one of the most commonly
used treatments for alopecia areata. 3. Anthralin: This medication is applied to the skin and can
help reduce inflammation and promote hair growth


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:: Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors.
Here are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These can help suppress the immune system and promote hair growth. They are usually
administered every 4-6 weeks. 2. Topical corticosteroids: These can be applied directly to the
affected area to reduce inflammation and promote hair growth. 3. Minoxidil (Rogaine): This is a
solution that is applied to the scalp to stimulate hair growth and slow down hair loss. 4. Anthralin
(Psoralen): This is a medication that can be applied to the skin to reduce inflammation and promote
hair growth. 5. Phototherapy: Exposure to ultraviolet light can help reduce inflammation and promote
hair growth. 6. Immunotherapy: This involves using injections or topical applications of substances
that stimulate the immune system to attack cancer cells. 7. Platelet-rich plasma (PRP


4. Custom Par

Llama.generate: prefix-match hit




Response #4:: Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors.
Here are some effective treatments and solutions for addressing this condition:  1. Corticosteroid
injections: These can help suppress the immune system and promote hair growth. They are usually
administered every 4-6 weeks. 2. Topical corticosteroids: These can be applied directly to the
affected area to reduce inflammation and promote hair growth. 3. Minoxidil (Rogaine): This is a
solution that is applied to the scalp to stimulate hair growth and slow down hair loss. 4. Anthralin
(Psoralen): This is a medication that can be applied to the skin to reduce inflammation and promote
hair growth. 5. Phototherapy: Exposure to ultraviolet light can help reduce inflammation


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:Please explain the difference between male pattern baldness and alopecia areata. Also, provide a few
preventive measures that can help reduce the risk of sudden patchy hair loss?  Answer:  Sudden
patchy hair loss, commonly seen as localized bald spots on the scalp, can be caused by several
factors. The possible causes include:  1. Alopecia areata: This is an autoimmune condition that
causes hair loss in patches. It can occur at any age and affects both men and women. 2. Male pattern
baldness: This is a common cause of hair loss in men, caused by the conversion of testosterone to
dihydrotestosterone (DHT) in the body. DHT causes hair follicles to shrink, leading to hair loss. 3.
Telogen effluvium: This is a condition where there is a sudden increase in the number of hair
follicles that stop growing and enter the resting phase. This can cause hair to fall out in patches.
4. Stress: Prolonged stress can cause hair loss due to the release of hormones such as cortisol,
which c

### ✅ Best Response

## **Best = Prompt #4 -> Response #5**

**Parameter combo:** `max_tokens=300, temperature=1.0, top_p=0.7, top_k=75`

### Why it’s the best

This is the strongest overall because it covers **BOTH** halves of the question better than the others:

1. Covers multiple possible causes (good differential diagnosis):

	* autoimmune (alopecia areata)
	* hormonal imbalance
	* stress
	* infections (fungal like ringworm)
	* skin conditions
	* nutrient deficiencies
	* genetics

2. **Matches treatments to the underlying cause:** Instead of assuming everything is alopecia areata, it correctly implies that treatment depends on the cause, which is important and patient-safe.

3. **More realistic + clinically useful:** Patchy hair loss can look the same visually but have very different causes — this answer acknowledges that.

---

# ❌ Worst Response

## **Worst = Prompt #2 -> Response #2**

**Parameter combo:** `max_tokens=175, temperature=0.4, top_p=0.5, top_k=1`

### Why it’s the worst

This is the weakest because it contains a major factual errors and risks misleading the patient:

1. **Incorrect medication labeling:** It says: **“Anthralin (Psoralen)”** but
Anthralin and Psoralen are **not the same medication**:

	* **Anthralin** = topical irritant sometimes used for alopecia areata
	* **Psoralen** = used with UVA light therapy (PUVA), a different therapy approach

    That kind of mistake is a **reliability red flag** in medical content.

2. **Oversimplifies the diagnosis:** It assumes patchy hair loss is alopecia areata without seriously considering alternatives (fungal infection, traction, trichotillomania, etc.).

3. **Mid-level completeness:** It gives a partial treatment list but doesn’t strongly cover causes or safety guidance.


## Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [32]:
varied_prompts_func_and_params(sys_promp, response, user_query4)



Using Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and...

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:Answer:  There are several treatment options available for individuals who have sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the severity and location of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, swelling, and inflammation. 2. Rehabilitation
therapy: To help regain lost function and improve cognitive and physical abilities. This may include


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:Answer:  There are various treatments that may be recommended for a person who has sustained a
physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.
The specific treatment plan will depend on the nature and severity of the injury, as well as the
individual needs and circumstances of the patient. Some possible treatments may include:  1.
Medications: To manage symptoms such as pain, inflammation, and anxiety. 2. Rehabilitation therapy:
To help regain lost function and improve cognitive and physical abilities. This may include physical
therapy, occupational therapy, speech therapy, and cognitive therapy. 3. Surgery: To repair damaged
brain tissue or relieve pressure on the brain. 4. Lifestyle


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:Answer:  There are several treatment options available for individuals who have sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the severity and location of the injury, as well as the
individual's overall health and medical history. Here are some common treatments that may be
recommended:  1. Medications: Depending on the type and severity of the injury, medications such as
pain relievers, anti-seizure drugs, and antidepressants may be prescribed to manage symptoms and
promote healing. 2. Rehabilitation therapy: Physical, occupational, and speech therapy may be
necessary to help the individual regain lost function and improve their quality of life. 3. Surgery:
In some cases, surgery may be required to repair damaged brain tissue or relieve pressure on the
brain. 4. Lifestyle changes: Changes to diet, exercise, and sleep habits may be recommended to help
the individual

Llama.generate: prefix-match hit




Response #4:Answer:  There are several treatment options available for individuals who have sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the severity and location of the injury, as well as the
individual's overall health and medical history. Here are some common treatments that may be
recommended:  1. Medications: Depending on the type and severity of the injury, medications such as
pain relievers, anti-seizure drugs, and antidepressants may be prescribed to manage symptoms and
promote healing. 2. Rehabilitation therapy: Physical, occupational, and speech therapy may be
necessary to help the individual regain lost function and improve their quality of life. 3. Surgery:
In some cases, surgery may be required to repair damaged brain tissue or relieve pressure on the
brain.


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:What is the typical recovery time frame for such injuries, and what can be done to improve recovery
outcomes? Answer: There are several treatment options for individuals who have sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
specific treatment plan will depend on the severity and location of the injury, as well as the
individual's overall health and medical history. Some common treatments for brain injuries include:
1. Medications: To manage symptoms such as pain, swelling, and inflammation. 2. Rehabilitation
therapy: To improve cognitive, physical, and emotional functioning. This may include occupational
therapy, speech therapy, and physical therapy. 3. Surgery: In some cases, surgery may be necessary
to relieve pressure on the brain or repair damaged tissue. 4. Lifestyle changes: Such as avoiding
activities that exacerbate the injury, getting regular exercise, and practicing stress-reducing
techniques 

### ✅ Best Response

## **Best = Prompt #1 -> Response #5**

**Parameter combo:** `max_tokens=300, temperature=1.0, top_p=0.7, top_k=75`

### Why it's the best

This is the most complete, realistic, and medically relevant for a patient:

1. Covers the real “big buckets” of TBI treatment:

	* **Medications** (and it gives meaningful examples like seizures + blood pressure support, not just “pain”)
	* **Surgery** (relieving pressure, clots, repair)
	* **Rehabilitation therapy** (PT/OT/speech/cognitive therapy)
	* **Lifestyle modifications** (rest + avoiding worsened injury)

2. Feels like an actual medical plan. It separates **acute management** (stabilization + surgery/meds) from **recovery** (rehab), which is exactly what patients need to understand.

3. Good detail without being too technical. It stays understandable while still being informative.

---

# ❌ Worst Response

## **Worst = Prompt #1 -> Response #1**

**Parameter combo:** `max_tokens=128, temperature=0, top_p=0.95, top_k=50`

### Why it's the worst

This one is the least helpful and least complete:

1. Cuts off mid-answer. It doesn’t finish the rehab list (“This may include…”) and never expands.

2. Misses critical treatment categories. No mention of:

	* Surgery
	* Seizure prevention
	* ICU-level monitoring / preventing complications
	* Long-term rehab details/support

3. **Too generic/vague:** “Pain, inflammation, anxiety” is not wrong, but it’s weak and oversimplified for brain injury care.



## Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [33]:
varied_prompts_func_and_params(sys_promp, response, user_query5)



Using Prompt #1: You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and...

1. Default parameters: max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit



Response #1:: If someone has fractured their leg while hiking, it is essential to take immediate action to
prevent further injury and ensure proper care. Here are the necessary precautions and treatment
steps:  1. Stop activity: Immediately stop any physical activity to avoid exacerbating the injury.
2. Assess the situation: Evaluate the severity of the fracture and determine if there are any other
injuries or complications. 3. Call for help: Contact emergency services or seek assistance from a
qualified medical professional. 4. Immobilize the leg: Use


2. Custom Parameters: max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit




Response #2:: If someone has fractured their leg while hiking, it is essential to take immediate action to
prevent further injury and ensure proper care. Here are the necessary precautions and treatment
steps:  1. Stop activity: Immediately stop any physical activity to avoid exacerbating the injury.
2. Assess the situation: Evaluate the severity of the fracture and determine if there are any other
injuries or complications. 3. Call for help: Contact emergency services or seek assistance from a
trained medical professional. 4. Immobilize the leg: Use a splint or brace to immobilize the
affected leg, ensuring it remains stable and aligned. 5. Apply ice: Apply ice packs to reduce
swelling and pain. Wrap the ice in a


3. Custom Parameters: max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...



Response #3:As a reliable medical assistant, I would advise that if someone has fractured their leg during a
hiking trip, it is essential to seek immediate medical attention. The following are necessary
precautions and treatment steps:  1: Stop all activity and rest the affected leg: Continuing to walk
or engage in physical activity can worsen the injury and lead to further complications.  2:
Immobilize the affected leg: Use a splint or brace to immobilize the leg and prevent any further
movement. This will help reduce pain and promote healing.  3: Apply ice: Applying ice to the
affected area can help reduce swelling and relieve pain. Wrap an ice pack in a towel and apply it
for 15-20 minutes at a time, several times a day.  4: Elevate the affected leg: Elevating the leg
above the level of the heart can help reduce swelling and promote blood flow.  5: Monitor for signs
of infection: Keep an eye out for signs of infection, such as redness, swelling, or increased pain.
If you notice an

Llama.generate: prefix-match hit




Response #4:As a reliable medical assistant, I would advise that if someone has fractured their leg during a
hiking trip, it is essential to seek immediate medical attention. The following are necessary
precautions and treatment steps:  1: Stop all activity and rest the affected leg: Continuing to walk
or engage in physical activity can worsen the injury and lead to further complications.  2:
Immobilize the affected leg: Use a splint or brace to immobilize the leg and prevent any further
movement. This will help reduce pain and promote healing.  3: Apply ice: Applying ice to the
affected area can help reduce swelling and relieve pain. Wrap an ice pack in a towel and apply it
for 15-20 minutes at a time, several times a day.  4: Elevate the affected leg: Elevating the leg
above the level of the


5. Custom Parameters: max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit




Response #5:Precautions:  * Carefully immobilize the affected limb to prevent further injury * Apply ice packs
to reduce swelling * Keep the patient warm and dry * Monitor vital signs and watch for signs of
shock or infection  Treatment Steps:  * Call for emergency medical assistance as soon as possible *
Provide basic first aid, such as stopping bleeding and applying a clean dressing * Use a splint to
immobilize the affected limb * Apply a cast or brace to support the leg * Monitor the patient's
condition and adjust treatment as needed  Considerations for Care and Recovery:  * Ensure the
patient is comfortable and able to rest * Provide pain management as needed * Monitor the patient's
progress and adjust treatment as needed * Encourage the patient to follow a healthy diet and
exercise program to aid in their recovery * Consider physical therapy to help regain strength and
mobility in the affected limb.


### ✅ Best Response

## **Best = Prompt #3 -> Response #4**

**Parameter combo:** `max_tokens=200, temperature=0.1, top_p=0.3, top_k=25`

### Why it's the best

This response is the strongest because it includes **the most clinically important “field first-aid” guidance**, not just generic advice:

1. **Includes neurovascular checks:** It explicitly says to **assess sensation/movement/pulse** (neurovascular status). That’s *huge* for a broken leg outdoors because loss of circulation/nerve function is an emergency.

2. **Prioritizes immobilization correctly:** It recommends **splint/cast immobilization** (splinting is the key step before transport).

3. **Mentions pain control:** That matters for real-world comfort and prevents shock/panic.

4. Matches what a good first responder / clinic would say.
It reads like a real **5 step triage approach:**
    
      Step 1: **Stop activity**

      Step 2: **Assess danger signs**

      Step 3: **Immobilize**
      
      Step 4: **Manage pain**
      
      Step 5: **Get help**
---

# ❌ Worst Response

## **Worst = Prompt #5 -> Response #5**

**Parameter combo:** `max_tokens=300, temperature=1.0, top_p=0.7, top_k=75`

### Why it's the worst

1. Even though it gives some decent first-aid steps, it’s the worst overall because:

    * **It starts with a refusal / credibility collapse:** 'I am unable to provide information about specific medical conditions or treatments…'

    That’s *exactly the opposite* of what a patient needs in a potential emergency. It makes the user unsure whether to trust anything that follows.

2. It’s vague and incomplete for “care and recovery”.
It covers basic field care but doesn’t properly address:

	* **Urgent red flags** (bone sticking out, blue/cold foot, severe deformity, uncontrolled bleeding)
	* **Don’t move if displaced/open fracture**
	* **Getting evacuation / transport safely**
	* **Follow-up recovery considerations** (X-ray, casting vs surgery, rehab, DVT risk, weight-bearing guidance)

3. The tone is **non-committal:** Patients in emergencies need **clear, confident, action-oriented guidance** (with proper safety disclaimers), not “I can’t provide this, but…”


##Observations
*Note: Remember that responses #1-5 are actually just different parameter combinations that result in specific responses.*

##**Collected Data:**
---
Best Prompt/Response Combination:

* Prompt **#3** -> Response #4
* Prompt **#3** -> Response **#5**
* Prompt #2 -> Response **#5**
* Prompt #1 -> Response **#5**
* Prompt #4 -> Response **#5**

Worst Prompt/Response Combination:
* Prompt **#1** -> Response **#1**
* Prompt **#1** -> Response **#1**
* Prompt **#1** -> Response **#1**
* Prompt #2 -> Response #2
* Prompt #5 -> Response #5


##**Results**
---
###Best:
**Conclusions:** Prompt #3 was the best prompt for 2 out of the 5 prompts which was the most out of all of them. Therefore, prompt #3 is the best prompt. The #5 responses returned from parameter combination #5 was the best response for 4 out of the 5 of them. Therefore, the parameter combination that resulted in the #5 responses is the best of all.

**Prompt #3:** Serve as a medical assistant with expertise in evidence-based medicine, providing accurate responses and transparently indicating any limitations in knowledge.

**Response #5 (Params Combination):** max_tokens=300, temperature=1.0, top_p=0.7, top_k=75

Best combination: **Prompt #3 -> Response #5**

###Worst:

**Conclusions:** Prompt #1 was the worst prompt for 3 out of the 5 prompts which the most out of all of them. Therefore, prompt #1 is the worst prompt. The response returned from parameter combination #1 was the worst response for 3 out of the 5 of them. Therefore, the parameter combination that resulted in the #1 responses is the worst of all.

**Prompt #1:** You are a well-informed medical assistant who provides clear, accurate, and evidence-based healthcare responses. If there is not enough information to develop a sufficient and accurate answer then respond with “Not enough data for an acceptable answer.

**Response #1 (Params Combination):** max_tokens=128, temperature=0, top_p=0.95, top_k=50

Worst Combination: **Prompt #1 -> Response #1**

# Data Preparation for RAG

### Loading the Data

In [11]:
# Attempt to load the medical manual PDF using PyMuPDFLoader
try:
    data_load = PyMuPDFLoader(DATA_PATH)  # Initialize loader with the data path
    med_manual = data_load.load()         # Load all pages from the PDF
    print(f"Number of pages loaded: {len(med_manual)}")  # Output number of pages loaded
except Exception as e:
    # Handle loading failure: print the error and set a fallback value
    print(f"Failed to load PDF at {DATA_PATH}. Error: {e}")
    med_manual = ["ERROR: NO DATA LOADED"]

Number of pages loaded: 4114


### Data Overview

#### Checking number of pages and the content of the first 5 pages

In [34]:
# Print the total number of pages in the med_manual PDF
print(f"Number of pages in 'med_manual' pdf: {len(med_manual)}" + "\n")

# Print the content of pages 100 to 104 (inclusive) in the med_manual list
for i, page in enumerate(med_manual[100:105], start=1):
    print(f"--- Page {100+i} ---")  # Page number starts at 101 for display

    # Check if the page object has a 'page_content' attribute (common for objects)
    if hasattr(page, "page_content"):
        # If page_content is a string, strip whitespace and print it; otherwise, print as-is
        print(page.page_content.strip() if isinstance(page.page_content, str) else page.page_content, end="\n\n")
    # If page is a dictionary and has 'page_content' key
    elif isinstance(page, dict) and "page_content" in page:
        # If page_content is a string, strip whitespace and print it; otherwise, print as-is
        print(page["page_content"].strip() if isinstance(page["page_content"], str) else page["page_content"], end="\n\n")
    else:
        # For any other type, convert to string, strip whitespace, and print
        print(str(page).strip(), end="\n\n")

Number of pages in 'med_manual' pdf: 4114

--- Page 101 ---
Vitamin D is a prohormone with several active metabolites that act as hormones. Vitamin D is metabolized
by the liver to 25(OH)D, which is then converted by the kidneys to 1,25(OH)2D (1,25-
dihydroxycholecalciferol, calcitriol, or active vitamin D hormone). 25(OH)D, the major circulating form, has
some metabolic activity, but 1,25(OH)2D is the most metabolically active. The conversion to 1,25(OH)2D
is regulated by its own concentration, parathyroid hormone (PTH), and serum concentrations of Ca and
phosphate.
[
Table 4-6. Actions of Vitamin D and its Metabolites]
Vitamin D affects many organ systems (see Table 4-6), but mainly it increases Ca and phosphate
absorption from the intestine and promotes normal bone formation and mineralization. Vitamin D and
related analogs may be used to treat psoriasis, hypoparathyroidism, renal osteodystrophy, and possibly
leukemia and breast, prostate, and colon cancers; they may also be used fo

### Data Chunking

In [13]:
pdf_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, #Complete the code to define the chunk size
    chunk_overlap= 100 #Complete the code to define the chunk overlap
)

In [14]:
med_manual_chunks = data_load.load_and_split(pdf_splitter)

print(f"Number of pages loaded from 'med_manual' pdf: {len(med_manual)} pages" + "\n" + f"Number of chunks loaded from 'med_manual_chunks' pdf: {len(med_manual_chunks)} chunks\n")

print("\n====================================================================================================")
print(f"Chunk 105: {med_manual_chunks[105].page_content}) + \n")

print("\n====================================================================================================")
print(f"Chunk 106: {med_manual_chunks[106].page_content}) + \n")

print("\n====================================================================================================")
print(f"Chunk 107: {med_manual_chunks[107].page_content}) + \n")

Number of pages loaded from 'med_manual' pdf: 4114 pages
Number of chunks loaded from 'med_manual_chunks' pdf: 9214 chunks


Chunk 105: and charcoal), may occur during pregnancy. Anemia due to iron deficiency is common, as is anemia due
to folate deficiency, especially among women who have taken oral contraceptives. Vitamin D deficiency is
common during late pregnancy, predisposing the child to decreased bone mass.
Old age: Aging—even when disease or dietary deficiency is absent—leads to sarcopenia (progressive
loss of lean body mass), starting after age 40 and eventually amounting to a muscle loss of about 10 kg
(22 lb) in men and 5 kg (11 lb) in women. Undernutrition contributes to sarcopenia, and sarcopenia
accounts for many of the complications of undernutrition (eg, decreased nitrogen balance, increased
susceptibility to infections). Causes of sarcopenia include the following:
• Decreased physical activity
• Decreased food intake
• Increased levels of cytokines (particularly inter

### Embedding

In [15]:
# Initialize the embedding model with a specified model name
embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDED_MODEL_NAME)

content_1 = get_chunk_content(med_manual_chunks[105])
content_2 = get_chunk_content(med_manual_chunks[106])

embedding_1 = embedding_model.embed_query(content_1)
embedding_2 = embedding_model.embed_query(content_2)

print(f"Dimension of the embedding vector: {len(embedding_1)}")
if len(embedding_1) != len(embedding_2):
    print("Warning: The embeddings have different dimensions.")
else:
    print("Embeddings have the same dimensions.")

print("First 5 values of embedding 1:", embedding_1[:5])
print("First 5 values of embedding 2:", embedding_2[:5])

/tmp/ipython-input-1526996536.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDED_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimension of the embedding vector: 384
Embeddings have the same dimensions.
First 5 values of embedding 1: [0.008618833497166634, 0.013596496544778347, -0.053762998431921005, 0.10668179392814636, 0.051703520119190216]
First 5 values of embedding 2: [0.0747450441122055, -0.038403529673814774, -0.01743958331644535, 0.051027555018663406, -0.036169540137052536]


### Vector Database

In [16]:
# Ensure the vectorstore directory exists
os.makedirs(OUT_DIR, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=med_manual_chunks,
    embedding=embedding_model,
    persist_directory=OUT_DIR
)

vectorstore = Chroma(
    persist_directory=OUT_DIR,
    embedding_function=embedding_model
)

vectorstore.embeddings

# Total number of vectors
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")
# Sample similarity search for the given query
sample_query = "Headache"
top_k = 3
results = vectorstore.similarity_search(sample_query, k=top_k)
print(f"\nTop {top_k} results for query '{sample_query}':")
seen_contents = set()
unique_results = []
for doc in results:
    content = doc.page_content
    if content not in seen_contents:
        seen_contents.add(content)
        unique_results.append(doc)

# Replace the duplicate response with an additional unique (but valid) response if available
if len(unique_results) < top_k:
    # Fetch more results until results are unique, if possible
    more_results = vectorstore.similarity_search(sample_query, k=top_k + 5)
    for doc in more_results:
        content = doc.page_content
        if content not in seen_contents:
            seen_contents.add(content)
            unique_results.append(doc)
        if len(unique_results) >= top_k:
            break

for index, doc in enumerate(unique_results[:top_k], 1):
    preview = wrap_text(doc.page_content[:650].replace("\n", " ") + ("..." if len(doc.page_content) > 650 else ""))
    print(f"{index}. {preview}")

Total documents in vectorstore: 9214

Top 3 results for query 'Headache':
1. Chapter 178. Headache Approach to the Patient With Headache Headache is pain in any part of the
head, including the scalp, face (including the orbitotemporal area), and interior of the head.
Headache is one of the most common reasons patients seek medical attention. Pathophysiology Headache
is due to activation of pain-sensitive structures in or around the brain, skull, face, sinuses, or
teeth. Etiology Headache may occur as a primary disorder or be secondary to another disorder.
Primary headache disorders include migraine, cluster headache (including chronic paroxysmal
hemicrania and hemicrania continua), and tension-type headache. Secondary...
2. noted. [Table 178-2. Some Characteristics of Headache Disorders by Cause] Review of systems should
seek symptoms suggesting a cause, including • Vomiting: Migraine, increased intracranial pressure •
Fever: Infection (eg, encephalitis, meningitis, sinusitis) • Red ey

/tmp/ipython-input-2065548343.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


### Retriever

In [17]:
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Using similarity search
    search_kwargs={"k": 3}     # Retrieve top 3 similar documents
)

# Getting relevant documents and results for the queries
print(f"\n--- Retriever Validation ---")
if retriever:
    print("Retriever created successfully.")
    test_queries = ["Headache", "sepsis", "appendicitis", "hair loss"]
    for test_query in test_queries:
        try:
            # Fetch more results than top_k to enable replacement of duplicates if needed
            top_k = 3
            fetch_k = top_k + 5
            retrieved_docs = retriever.invoke(test_query)

            # If retriever returns fewer than fetch_k, get more if possible
            if len(retrieved_docs) < top_k:
                retrieved_docs = retriever.invoke(test_query)

            # De-duplicate the results: preserve order and get top_k unique ones
            seen_contents = set()
            unique_docs = []
            # Try first batch:
            for doc in retrieved_docs:
                content = doc.page_content
                if content not in seen_contents:
                    unique_docs.append(doc)
                    seen_contents.add(content)
                if len(unique_docs) >= top_k:
                    break
            # If not enough, try fetching more
            if len(unique_docs) < top_k:
                extra_docs = retriever.invoke(test_query)
                # Keep only new docs
                for doc in extra_docs:
                    content = doc.page_content
                    if content not in seen_contents:
                        unique_docs.append(doc)
                        seen_contents.add(content)
                    if len(unique_docs) >= top_k:
                        break
            #Prints the top 3 unique results for the query
            print(f"\nTop {len(unique_docs)} unique results for query '{test_query}':")
            if len(unique_docs) > 0:
                for i, doc in enumerate(unique_docs[:top_k], 1):
                    content = doc.page_content[:300].replace("\n", " ")
                    print(wrap_text(f"{i}. {content}{'...' if len(doc.page_content) > 300 else ''}"))
                if len(unique_docs) < top_k:
                    print(f"Warning: Only {len(unique_docs)} unique docs found for query '{test_query}'.")
            else:
                print(f"No results found for query '{test_query}'.")
        except Exception as e:
            print(f"Error during sample query with retriever: {e}")
else:
    print("Failed to create retriever; check vectorstore initialization.")


--- Retriever Validation ---
Retriever created successfully.

Top 3 unique results for query 'Headache':
1. Chapter 178. Headache Approach to the Patient With Headache Headache is pain in any part of the
head, including the scalp, face (including the orbitotemporal area), and interior of the head.
Headache is one of the most common reasons patients seek medical attention. Pathophysiology Headache
is due t...
2. noted. [Table 178-2. Some Characteristics of Headache Disorders by Cause] Review of systems
should seek symptoms suggesting a cause, including • Vomiting: Migraine, increased intracranial
pressure • Fever: Infection (eg, encephalitis, meningitis, sinusitis) • Red eye and/or visual
symptoms (halos, b...
3. to treatment; testing visual acuity is not sensitive enough to warn of impending vision loss.
Migraine Migraine is an episodic primary headache disorder. Symptoms typically last 4 to 72 h and
may be severe. Pain is often unilateral, throbbing, worse with exertion, and accompan

### RAG and Grounding/Relevance Evaluation Functions

In [22]:
#Generates response to the user's input, retrieving relevant documents for context and running the LLM.
def generate_rag_response(
    user_input,  # str: The user's query or input question to be answered.
    k_docs=3,    # int: Number of relevant document chunks to retrieve for context.
    max_tokens=300,  # int: Maximum number of tokens in the model's generated response.
    temperature=0,   # float: Sampling temperature for model response randomness.
    top_p=0.95,      # float: Top-p (nucleus) sampling value for controlling diversity.
    top_k=50         # int: Top-k sampling value for number of highest probability vocab tokens to keep for generation.
):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(user_input, k=k_docs)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Try to generate a response that answers the prompt
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'
    # Limit output to 650 characters and append '...' if truncated
    if len(response) > 650:
      limited_output = response[:650] + '...'
    else:
      limited_output = response
    wrap_output = wrap_text(limited_output)
    print(wrap_output)

#Generates a response to user_input using RAG, then evaluates and prints the groundedness and relevance scores.
def generate_ground_relevance_response(
    user_input,    # str: The user's input query to be answered and evaluated
    k_docs=3,      # int: Number of most relevant documents/chunks to retrieve (for RAG context)
    max_tokens=300,# int: Maximum tokens for the generated response
    temperature=0, # float: Sampling temperature for generation (0 = deterministic)
    top_p=0.95,    # float: Nucleus sampling parameter; model considers tokens with cumulative probability top_p
    top_k=50       # int: The number of highest probability vocabulary tokens to keep for top-k filtering
):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(user_input, k=k_docs)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and qna_system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""
    # Try to generate a response that answers the prompt
    try:
        response = llm(
                prompt=prompt,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                stop=['INST'],
                echo=False
                )
        answer =  response["choices"][0]["text"]
    except Exception as e:
        answer = f"Error generating answer: {e}"

    # Combine user_prompt and groundedness_rater_system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and relevance_rater_system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Try to generate a groundedness score for the model's answer
    try:
        response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
        )
        # Extract the groundedness rating text from the model's response
        text_1 = response_1['choices'][0]['text']
    except Exception as e:
        # Handle any errors encountered during LLM func call for groundedness
        text_1 = f"Error generating groundedness response 1: {e}"
        print(text_1)

    # Try to generate a relevance score for the model's answer
    try:
        response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
        )
        # Extract the relevance rating text from the model's response
        text_2 = response_2['choices'][0]['text']
    except Exception as e:
        # Handle any errors encountered during LLM func call for relevance
        text_2 = f"Error generating relevance response 2: {e}"
        print(text_2)

    # Limit output to 650 characters and append '...' if truncated
    if len(text_1) > 650:
      limited_output_1 = text_1[:650] + '...'
    else:
      limited_output_1 = text_1

    if len(text_2) > 650:
      limited_output_2 = text_2[:650] + '...'
    else:
      limited_output_2 = text_2

    # Print the results before returning
    print(wrap_text(limited_output_1), end="\n\n")
    print(wrap_text(limited_output_2))

# Question Answering & Fine Tuning using RAG

## Using the RAG response function with previously used LLM functions

**Output Description:** Each query is tested with 5 different parameter combinations. That means each query produces 5 total answers so there will be a total of 25 answers in the is section(5 x 5)

*Note: Remember that Answers #1-5 are actually just different parameter combinations that result in specific answers.*

Describing how the model outputs might be effected by parameter values of each combination when using: **varied_func_and_params(generate_rag_response, user_query#, FLAG)** and **generate_rag_response(user_query#, k_docs, max_tokens, temperature, top_p, top_k)**

---

1) **Default Parameters:** k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50, FLAG=True
  
    **Output Indications:** These parameters produce short, highly deterministic answers which may lack creativity or variation. A temperature of 0 means no randomness (the model gives the most likely next token at every step), top_p=0.95 allows common, high-probability words, and top_k=50 gives the model a moderate pool of possible words to choose from. With max_tokens=128 comes a lack of extremely creative or thorough responses, but it makes up for it by providing focused, concise responses. With k_docs=3, the response will be grounded in the top 3 retrieved documents or chunks.
---
2) **Custom Parameters:** k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1, FLAG=True

    **Output Indications:** Expect answers with only modest variability, still very focused and concise. Still uses 3 documents(k_docs), but allows a longer answers (max_tokens=175) than the default parameter max_token value of 128. This configuration slightly increases randomness with a temperature of 0.4, but using top_k=1 means the model will always pick the single most likely token (even though top_p is lower, it is redundant when top_k=1).
---
3) **Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100, FLAG=True

    **Output Indications:** Uses fewer documents (k_docs=2), with a much longer answer (max_tokens=250) which allows for a long output window, so the answers will be more creative and varied, but will lose focus or coherence. A higher temperature (0.7) increases randomness and diversity in the model's responses. The top_p=0.0 value effectively disables nucleus sampling, so only top_k=100 is used, which gives the model many token choices.
---
4) **Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25, FLAG=True

    **Output Indications:** Use these parameters for reliability, accuracy and relevance rather than creativity. Only uses a single relevant document(k_docs=1) for the model's grounding, which can increase risk of missing details but enhances focus. The max_tokens=200 value gives moderate answer length. The low temperature (0.1) value yields mostly deterministic outputs, but the top_p=0.3 value restricts the candidate pool to only the most probable tokens, usually resulting in concise, safe answers. The top_k=25 value moderately increases potential variability within the top tokens.
---
5) **Custom Parameters:** k_docs=2, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75, FLAG=True

    **Output Indications:** Leverages 4 documents(k_docs=4) for this model's grounding which provides the largest amount of data out of all the project's model parameter configurations. With the highest temperature (1.0), outputs are at their most diverse and creative, though possibly less reliable. The top_p=0.7 value and the top_k=75 value both limit the answer space, encouraging some focus but still allowing a lot of variation. The largest max_tokens(300) value yet results in the most thorough or elaborate answers, excellent for brainstorming or open-ended questions.



## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [68]:
#Note: Remember that Answers #1-5 are actually just different parameter combinations that result in specific answers.
varied_func_and_params(generate_rag_response, user_query1, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following protocol for
managing sepsis in a critical care unit:  1. Early recognition and aggressive management of sepsis
are crucial to preventing severe organ dysfunction and death. 2. Monitor patients for signs of
sepsis, including fever, tachycardia, tachypnea, and altered mental status. 3. Measure serum lactate
levels and perform blood cultures to confirm the presence of infection. 4. Administer broad-spectrum
antibiotics promptly, based on the suspected pathogen and the patient's medical history and
allergies. 5. Provide appropriate fluid resuscitation with crystal...


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following protocol for
managing sepsis in a critical care unit:  1. Early recognition and aggressive management of sepsis
are crucial to preventing severe organ dysfunction and death. 2. Monitor patients for signs of
sepsis, including fever, tachycardia, tachypnea, and altered mental status. 3. Measure serum lactate
levels and perform blood cultures to confirm the presence of infection. 4. Administer broad-spectrum
antibiotics promptly, based on the suspected pathogen and the patient's medical history and
allergies. 5. Provide appropriate fluid resuscitation with crystal...


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following protocol for
managing sepsis in a critical care unit:  1. Early recognition and aggressive management of sepsis
are crucial to preventing severe organ dysfunction and death. 2. Measure vital signs, including
temperature, blood pressure, tachycardia, and tachypnea, every 4-6 hours or more frequently if
clinically indicated. 3. Monitor fluid status closely, including daily weights, urine output, and
fluid balance. 4. Administer broad-spectrum antibiotics as soon as sepsis is suspected, and continue
until the patient's cultures are negative and they have been afeb...


4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following protocol for
managing sepsis in a critical care unit:  1. Early recognition: Sepsis should be suspected in any
critically ill patient with signs of infection, such as fever, tachycardia, or tachypnea. 2. Rapid
administration of antibiotics: Broad-spectrum antibiotics should be administered promptly, ideally
within the first hour of recognition of sepsis. 3. Fluid resuscitation: Crystalloid fluids should be
administered to maintain mean arterial pressure (MAP) ≥ 65 mmHg and urine output ≥ 0.5 mL/kg/h. 4.
Vasopressor support: Vasopressors may be needed to maintai...


5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75
Sorry, I encountered the following error:   Requested tokens (2542) exceed context window of 2300


### ✅ Best answer: **Answer 4**

**Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### Why it’s the best

Answer 4 is the strongest because it includes **core, high-priority ICU sepsis actions** in a way that’s medically relevant and practical:

1. **Recognizes sepsis early** (critical because delays increase mortality)
2. **Emphasizes antibiotics quickly** (“within the first hour” is the widely taught standard for suspected septic shock / severe cases)
3. **Includes fluid resuscitation** and gives **clear targets**:

	* MAP ≥ 65 mmHg
	* Urine output ≥ 0.5 mL/kg/hr
  
4. **Introduces vasopressors** appropriately as the next step if fluids aren’t enough. Even though it’s cut off at the end, it’s still the most *correct and clinically structured* of the available options.

---

## ❌ Worst answer: **Answer 3**

**Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

### Why it’s the worst

Answer 3 contains multiple things that would be **poor-quality or misleading** for a patient asking about ICU protocol:

1. **Bad monitoring frequency**

	* It says vitals “every 4–6 hours.”
	* In actual critical care sepsis management, monitoring is usually **continuous or very frequent**, especially early on (blood pressure, oxygenation, urine output, mental status, lactate trends, etc.).
	* “Every 4–6 hours” seriously underplays urgency and intensity.

2. **Incorrect/oversimplified antibiotic guidance**

	* It says continue antibiotics “until cultures are negative and they have been not feverish…”
	* That’s **not how antibiotic duration is decided**.
	* Cultures can be negative even in true sepsis, and fever resolution alone is not a safe stopping rule. Duration depends on **source control, organism (if found), response, imaging, and clinical course**.

3. **Doesn’t include key ICU protocol pieces**

	* No mention of vasopressors (unless later)
	* No lactate-driven reassessment
	* Doesn’t mention source control clearly

---

## 🚫 **Unacceptable answers:** Answer #5

## Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [69]:
varied_func_and_params(generate_rag_response, user_query2, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit


Answer: The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by
brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower
quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and
rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line
joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right
lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from
passive extension of the right hip joint that stretches the ili...


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit


Answer: The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by
brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower
quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and
rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line
joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right
lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from
passive extension of the right hip joint that stretches the ili...


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


Llama.generate: prefix-match hit


Please answer based on the information provided in the Merck Manual of Medical Diagnosis and
Therapy, Nineteenth Edition.


4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit


Please answer based on the information provided in the Merck Manual of Medical Diagnosis and
Therapy, Nineteenth Edition.


5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75
Sorry, I encountered the following error:   Requested tokens (2581) exceed context window of 2300


### ✅ Best answer: **Answer 1**

**Default Parameters:** k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50

### Why it's the best

Answer 1 is the strongest medical response because it gives the most **clinically accurate and high-value description of appendicitis symptoms**, using classic presentation details that doctors actually rely on:

1. **Correct pain progression:** starts **periumbilical/epigastric**, then **moves to the right lower quadrant** (very characteristic of appendicitis).
2. **Includes key associated symptoms:** nausea, vomiting, and loss of appetite (anorexia).
3. **Explains worsening with movement/cough:** a useful real-world clue that increases suspicion.
4. **Includes major physical exam signs:**

	* **McBurney’s point tenderness**
	* **Rebound tenderness**
	* **Rovsing sign**
	* Mentions pain related to hip extension (suggestive of irritation of nearby muscles)

Even though it doesn’t fully finish the “medicine vs surgery” part due to truncation, it still contains the **highest-quality, most diagnostic medical information** of the set.

---

## ❌ Worst answer: NONE

Answer 2 is NOT worst — the real “worst” among acceptable ones doesn’t exist. Because **Answers 3, 4, and 5 are unacceptable**, so there’s no fully valid answer left that is clearly 'worst' while still being an actual medical answer.


---

## 🚫 **Unacceptable answers:** Answer #3, Answer #4, Answer #5


## Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [70]:
varied_func_and_params(generate_rag_response, user_query3, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit


Please provide accurate information based on the Merck Manual of Medical Diagnosis and Therapy, 19th
Edition.


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit


Please provide accurate information based on the Merck Manual of Medical Diagnosis and Therapy, 19th
Edition.


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...

Please provide a concise answer based on the information available in the Merck Manual of Medical
Diagnosis and Therapy, 19th Edition.


4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit


Answer: Based on the information available in the Merck Manual of Medical Diagnosis and Therapy,
Nineteenth Edition, there are several effective treatments for sudden patchy hair loss, also known
as localized bald spots on the scalp. These include:  1. Medications: Minoxidil (Rogaine) and
finasteride (Propecia) are two medications that have been approved by the FDA for treating male
pattern baldness and can be effective in treating sudden patchy hair loss as well. 2. Platelet-rich
plasma (PRP) therapy: This is a non-surgical treatment that involves injecting platelet-rich plasma
into the scalp to stimulate hair growth. 3. Low-level laser ther...


5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit


Answer: Sudden patchy hair loss can be caused by a variety of factors, including hormonal
imbalances, autoimmune disorders, infections, and certain medications. Treatment options will depend
on the underlying cause. For example, if the hair loss is due to an autoimmune disorder such as
alopecia areata, treatment may involve steroid injections or topical creams to suppress the immune
system. If the hair loss is caused by a hormonal imbalance, treatment may involve hormone
replacement therapy. In some cases, medications such as minoxidil (Rogaine) or finasteride
(Propecia) may be prescribed to promote hair growth. It is important to consult a d...


### ✅ Best answer: **Answer 5**

**Custom Parameters:** k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75

### Why it’s the best

1. **Addresses BOTH parts of the question**: gives **possible causes** (autoimmune, hormonal, infections, meds) *and* **treatment direction**
2. Mentions the **most common “patchy bald spot” cause** conceptually: **alopecia areata (autoimmune)** and includes **typical first-line treatments** (steroid injections/topicals)
3. Gives a **safe next step**: recommends seeing a dermatologist (important because causes need confirmation)

### What lowers its quality (but still best here)

1. It’s **over-general** and mixes conditions:

	* Mentions **finasteride**, which is mainly for **male pattern hair loss**, not the classic “patchy bald spots” pattern (alopecia areata).
	* Hormone replacement therapy is also too broad and could be misleading without proper testing.

---

## ❌ Worst answer: **Answer 4**

**Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### Why it’s the worst

1.  **Misleads the patient toward the wrong diagnosis/treatment:**

	* It pushes **minoxidil + finasteride** as if they apply broadly to sudden patchy hair loss.
	* That combo fits **androgenetic alopecia**, not classic **localized bald patches** (alopecia areata or fungal infection, etc.)

2. **Doesn’t explain causes clearly**, so the patient doesn’t know what might be behind the bald spots (autoimmune vs fungal vs traction vs thyroid, etc.)
3. Tries to anchor to the Merck Manual, but the content presented is **not well-targeted** to the patchy-bald-spot scenario.


---

## 🚫 **Unacceptable answers:** Answer #1, Answer #2, Answer #3

## Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [19]:
varied_func_and_params(generate_rag_response, user_query4, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50
Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following treatments for
patients with physical injuries to brain tissue that result in temporary or permanent impairment of
brain function:  1. Supportive care: Preventing systemic complications due to immobilization,
providing good nutrition, and preventing pressure ulcers. 2. Rehabilitation therapy: Evaluation by
rehabilitation specialists to establish baseline findings, followed by reevaluation before starting
rehabilitation therapy, and cognitive therapy for patients with severe cognitive dysfunction. 3.
Physical and occupational therapy: Modestly improving function...


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends the following treatments for
patients with physical injuries to brain tissue that result in temporary or permanent impairment of
brain function:  1. Supportive care: Preventing systemic complications due to immobilization,
providing good nutrition, and preventing pressure ulcers. 2. Rehabilitation therapy: Evaluation by
rehabilitation specialists to establish baseline findings, followed by reevaluation before starting
rehabilitation therapy, and cognitive therapy for patients with severe cognitive dysfunction. 3.
Physical and occupational therapy: Modestly improving function...


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


Llama.generate: prefix-match hit


Please provide accurate and reliable information based on the Merck Manual of Medical Diagnosis and
Therapy, 19th Edition.


4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit


Answer: The Merck Manual of Medical Diagnosis and Therapy recommends early intervention by
rehabilitation specialists for individuals with traumatic brain injuries (TBI). This intervention
should include prevention of secondary disabilities, such as pressure ulcers and joint contractures,
as well as cognitive therapy to address memory loss, problem-solving skills, language, and visual
disturbances. The specific rehabilitation therapy will depend on the level and extent of the injury,
with complete transsection causing flaccid paralysis and partial transsection causing spastic
paralysis. Immobilization of the affected area is also recommended ...


5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit


Please answer based on information provided in the Merck Manual of Medical Diagnosis and Therapy,
19th Edition.


### ✅ Best answer: **Answer 2**

**Custom Parameters:** k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1

### Why it’s the best

1. **Clinically appropriate focus:** emphasizes **supportive care** (nutrition, preventing pressure ulcers/immobility complications) and **rehabilitation**—which *is* a core part of recovery after brain injury.
2. Mentions **multidisciplinary rehab**, including **cognitive therapy** for severe dysfunction, and **PT/OT** for functional improvement.
3. **Patient-safe framing:** not pushing risky or incorrect treatments.

### What’s missing / weaker

1. It’s **incomplete/truncated**, and doesn’t cover key acute/ICU steps often needed in brain injury care (airway/breathing, intracranial pressure concerns, seizure management, imaging/surgical evaluation, etc.).
2. It reads like a rehab section rather than a full “recommended treatments” overview.

---

## ❌ Worst answer: **Answer 4**

**Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### Why it’s the worst

1. It mixes in **spinal cord injury concepts** (“complete transsection → flaccid paralysis,” “partial transsection → spastic paralysis”).
  That’s not the typical way to describe **brain tissue injury**, so it risks **confusing/misleading** a patient.
2. “Immobilization of the affected area” is **vague** and not clearly applicable to brain injury (brain tissue isn’t “immobilized” the way a limb is).
3. While it includes useful rehab themes, the **content drift** lowers reliability.

---

## 🚫 **Unacceptable answers:** Answer #3, Answer #5

## Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [20]:
varied_func_and_params(generate_rag_response, user_query5, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit


Answer: Based on the information provided in the Merck Manual of Diagnosis & Therapy, 19th Edition,
the necessary precautions and treatment steps for a person who has fractured their leg during a
hiking trip are as follows:  1. Immobilization: The affected leg should be immobilized using a
splint or cast to prevent further injury and promote healing. 2. Rest, ice, compression, and
elevation (RICE): The patient should be advised to rest and avoid putting weight on the affected
leg. Ice and compression should be applied to reduce swelling, and the leg should be elevated to
reduce pain and promote blood flow. 3. Pain management: Pain should be m...


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit


Answer: Based on the information provided in the Merck Manual of Diagnosis & Therapy, 19th Edition,
the necessary precautions and treatment steps for a person who has fractured their leg during a
hiking trip are as follows:  1. Immobilization: The affected leg should be immobilized using a
splint or cast to prevent further injury and promote healing. 2. Rest, ice, compression, and
elevation (RICE): The patient should be advised to rest and avoid putting weight on the affected
leg. Ice and compression should be applied to reduce swelling, and the leg should be elevated to
reduce pain and promote blood flow. 3. Pain management: Pain should be m...


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


...

Answer: Based on the information provided in the Merck Manual of Medical Diagnosis and Therapy,
Nineteenth Edition, the necessary precautions and treatment steps for a person who has fractured
their leg during a hiking trip are as follows:  1. Immobilization: The leg should be immobilized
using a splint or cast to prevent further injury and promote healing. 2. Rest: The patient should
rest and avoid putting weight on the affected leg to minimize pain and swelling. 3. Ice: Apply ice
to the affected area for 15-20 minutes at a time, several times a day, to reduce pain and swelling.
4. Compression: Use an elastic bandage or compression dressing ...


4. Custom Parameters: k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25


Llama.generate: prefix-match hit


Answer: Based on the information provided in the Merck Manual of Medical Diagnosis and Therapy,
Nineteenth Edition, the necessary precautions and treatment steps for a person who has fractured
their leg during a hiking trip are as follows:  1. Rest, ice, and compression: The affected leg
should be immobilized with a splint or cast to prevent further injury and promote healing. Ice
should be applied to the area to reduce swelling and pain, and compression should be used to help
reduce bleeding and inflammation. 2. Stretching and strengthening exercises: After the initial acute
phase of the injury has passed, gentle stretching and strengthening...


5. Custom Parameters: k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75


Llama.generate: prefix-match hit


Answer:  1. Rest, ice, compression, and elevation (RICE) should be applied immediately to reduce
pain and swelling. 2. Splinting or immobilization may be necessary to prevent further injury and
promote healing. 3. Treatment of life- or limb-threatening injuries should be sought immediately,
including hemorrhagic shock and arterial injuries. 4. Nerve conduction studies may be indicated for
nerve injuries. 5. Closed reduction is usually maintained by casting or splinting, while open
reduction may require surgical hardware. 6. Patients at higher risk of deep vein thrombosis (DVT)
should receive additional preventive treatment, such as low-dose


### ✅ Best Answer: **Answer 5**

**Custom Parameters:** k_docs=4, max_tokens=300, temperature=1.0, top_p=0.7, top_k=75

### Why it’s the best

It’s the most **complete and medically useful** for a real hiking fracture situation:

1.  **Good emergency priorities**

	* Includes **RICE + immobilization** early, which is correct first-aid advice.
	* Mentions urgent red flags: **life- or limb-threatening injuries** like hemorrhage/shock/arterial injury.

2. **More realistic “real-world fracture care”**

	* Mentions **closed reduction vs open reduction surgery**, which is accurate for how fractures are stabilized.
	* Brings up **DVT prevention**, which is an important recovery risk after leg fractures (especially immobilization/surgery).

---

# ❌ Worst Answer: **Answer 4**

**Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### Why it’s the worst

1. It introduces a **potentially unsafe or misleading steps: **

	* Encouraging “stretching and strengthening” right after a fracture can be inappropriate unless the fracture is stabilized and a clinician has cleared activity.
	* Early movement can worsen pain, displace the fracture, or delay healing.
	* Recommends stretching/strengthening exercises too early.

2. It still includes immobilization/ice/compression, but that exercise recommendation makes it the weakest.

3. “Compression should be used to reduce bleeding” isn’t usually a standard fracture goal unless there’s an open wound—compression has to be used carefully to avoid restricting circulation.

---

## 🚫 **Unacceptable answers:** NONE

##Observations

---
**Results:**

Best Answers: **5, 5,** 1, 2, 4

Worst Answers: **4, 4, 4**, 1, NONE

Unacceptable Answers: NONE, 1, 2, **3, 3, 3**, 4, **5, 5, 5**

---

**Conclusions:**

Answer #5 was the best response for 2 of the queries which is the most out of all of them. Therefore, Answer #5 is the **best overall response**.  Answer #4 appears 3 times as the worst response which makes it the **worst overall response**. Answer #3 and #5 both appear 3 times in as **unacceptable responses** which makes them tied for the most number of overall unacceptable responses.

Best Answer: **Answer #5**

Worst Answer: **Answer #4**

Greatest Number of Unacceptable Answers: **Answer #3** and **Answer #5**

# Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [23]:
varied_func_and_params(generate_ground_relevance_response, user_query1, FLAG)


1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit


Error generating groundedness response 1: Requested tokens (2408) exceed context window of 2300
Error generating relevance response 2: Requested tokens (2369) exceed context window of 2300
Error generating groundedness response 1: Requested tokens (2408) exceed context window of 2300

Error generating relevance response 2: Requested tokens (2369) exceed context window of 2300


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided information, I would rate the answer as follows:

Based on the provided information, I would rate the relevance of the answer as 4 - High relevance.
The response addresses most key aspects of the question and provides specific information about the
protocol for managing sepsis in a critical care unit, including the use of


3. Custom Parameters: k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context and answer, I would rate the groundedness as follows:  ###Answer:  1.
Suspect sepsis in patients with signs of severe infection, such as fever, tachycardia, tachypnea,
confusion, or hypotension. 2. Obtain cultures of blood and other appropriate specimens to confirm
the diagnosis of bacteremia or sepsis. 3. Initiate empiric antibiotics after obtaining appropriate
cultures, using broad-spectrum agents effective against common causes of sepsis, such as Gram-
positive and Gram-negative bacteria, and fungi. 4. Monitor and adjust antibiotics according to the
results of culture and susceptibility testing. 5. Surgically...

Based on the provided context and the answer given, I would rate the relevance as follows:
###Rating(1-5): 4 - High relevance:  The answer directly addresses the user's question by providing
a protocol for managing sepsis in a critical care unit. The response covers most key requested
details, such as obtaining cultures, initiating empiric anti

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context and answer, I would rate the groundedness of the response as follows:
Rating: 4 - Mostly grounded.  The main points of the answer are supported by the context, but there
are some minor unsupported details. The answer provides a clear and accurate summary of the protocol
for managing sepsis in a critical care unit based on the Merck Manual of Medical Diagnosis and
Therapy. However, some specific details, such as the exact frequency of monitoring and testing, are
not explicitly stated in the context and may be considered unsupported additions. Additionally,
there is no mention of any contradictions or hallucinat...

Based on the provided information, I would rate the relevance of the answer as a 4 out of 5. The
response directly addresses the user's question by providing a detailed overview of the protocol for
managing sepsis in a critical care unit, including early recognition, administration of antibiotics,
and aggressive supportive care. The answer also r

### ✅Best Scores: **#4**

**Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### 📊Scoring Summaries:

**Scoring for #4**
* Groundedness (Rating 4 — Mostly grounded): The response stays closely aligned with the Merck-based context by covering core sepsis protocol steps (early recognition, antibiotics, fluids/vasopressors). A few details feel slightly “filled in” (like exact monitoring/testing frequency or thresholds) that aren’t clearly stated in the provided context, which keeps it from being fully grounded.

* Relevance (Rating 4 - High relevance): The answer directly targets what the user asked by outlining the critical-care sepsis workflow and prioritizing the key actions (recognize fast, treat fast, support organ function). It’s missing some helpful specificity (example criteria, antibiotic selection principles, or bundle timing), so it’s highly relevant but not maximally complete.

---

### ⚠️ Other Scores: None

### 🎯 Partial Scoring Response(s):

* **#2:** Revelance score of 4. No groundedness score.
* **#3:** Revelance score of 4. No groundedness score.

### 🚫 Invalid Response(s): **#1**, **#5**


## Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [24]:
varied_func_and_params(generate_ground_relevance_response, user_query2, FLAG)

1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Not grounded:

Based on the provided context and the user's question, I would rate the relevance of the answer as 4
out of 5. The answer provides a comprehensive overview of appendicitis, including its symptoms,
signs, diagnosis, treatment options, and other relevant information.


2. Custom Parameters: k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Not grounded: The answer
contains statements that are not supported by the context, such as "Treatment of acute appendicitis
is open or laparoscopic appendectomy." While the context does mention surgical removal of the
appendix, it does not specify the type of surgery. Therefore, this statement is not grounded in the
provided context. * Weakly grounded: The answer also contains statements that are only partially
supported by the context, such as "IV fluids and antibiotics." While the context does mention the
use of IV fluids and antibiotics in

Based on the provided context and the user's question, I would rate the relevance of the answer as 4
out of 5. The answer provides a comprehensive overview of appendicitis, including its symptoms,
signs, diagnosis, treatment, and other relevant information. However, the answer does not directly
address the user's specific question about the common symptoms for appendicitis and

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided information, I would rate the answer as follows:  ###Rating(1-5): 4 - Mostly
grounded  The answer provides a detailed description of the common symptoms and signs of
appendicitis, as well as the diagnostic criteria and treatment options. The information is supported
by the context provided in the Merck Manual of Medical Diagnosis and Therapy, which is a reputable
medical reference source. However, there are some minor unsupported details, such as the mention of
"atypical symptoms" and "rare instances," which are not explicitly supported by the context.
Therefore, I would rate the answer as mostly grounded, with a score...

Based on the provided context and answer, I would rate the relevance as follows:  ###Rating(1-5): 4
- High relevance  The answer provides a detailed response to the user's question, addressing all key
aspects of appendicitis, including its symptoms, diagnosis, and treatment options. The information
is accurate and relevant to the user's query, w

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided information, I would rate the answer as follows:  ###Rating(1-5): 4 - Mostly
grounded  The answer provides a detailed description of the common symptoms and signs of
appendicitis, which is well supported by the context. The answer also mentions the etiology of
appendicitis, which is indirectly supported by the context. However, there are some minor
unsupported details, such as the mention of "worms" as a possible cause of obstruction, which are
not explicitly mentioned in the context. Overall, the answer is well-grounded and provides a
accurate summary of the information available in the provided context.

Based on the provided information, I would rate the relevance of the answer as a 4 out of 5. The
response directly addresses the user's question by providing a detailed list of common symptoms for
appendicitis and the appropriate surgical procedure to treat it. The answer also includes additional
information about the etiology of appendicitis, which is relevant 

### ✅Best Scores: **#3** == **#4**

**#3 Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

**#4 Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### 📊Scoring Summaries:

**Scoring for #3**
* Groundedness (Rating 4 — Mostly grounded): The response is largely supported by the Merck Manual context because it accurately lists classic appendicitis symptoms/signs and the standard treatment approach. It loses a bit of groundedness by adding small extra statements (like “atypical symptoms” or “rare instances”) that aren’t clearly backed by the provided context.

* Relevance (Rating 4 — High relevance): The answer directly addresses what the user asked—symptoms, whether medicine can cure it, and the appropriate surgical treatment—so it stays on-topic and useful. It’s not a perfect 5 because a couple details could be clearer/more complete (like briefly explaining certain clinical signs or tightening the medicine vs surgery explanation).

**Scoring for #4**
* Groundedness (Rating 4 — Mostly grounded): The response is mostly grounded because it accurately lists common appendicitis symptoms and generally matches standard medical descriptions from the provided context. However, it adds a minor unsupported detail (like “worms” causing obstruction) that wasn’t clearly stated in the context, which slightly reduces groundedness.

* Relevance (Rating 4 — High relevance):The response is highly relevant because it directly answers the patient’s main questions by describing symptoms and stating that treatment typically requires surgery. Still, it falls short of being fully complete because it doesn’t clearly explain medicine vs. surgery (when antibiotics may apply) or outline the surgical procedure steps/details the patient asked for.
---
### ⚠️ Other Scores:

* **#1:** Groundedness score of 1. Revelance score of 4.
* **#2:** Groundedness score of 1. Revelance score of 4.

### 🎯 Partial Scoring Response(s): None

### 🚫 Invalid Response(s): **#5**

## Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [25]:
varied_func_and_params(generate_ground_relevance_response, user_query3, FLAG)

1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided information, I would rate the answer as follows:  * Factual claims that are
explicitly stated in the context: 4/5 (mostly grounded)       + The main points of the answer, such
as the effective treatments for alopecia areata, are supported by the context.    + However, some
minor details, such as the specific medications and their dosages, are not explicitly stated in the
context. * Factual claims that are not clearly supported by the context: 1/5 (not grounded)
+ The answer mentions "hormonal modulators such as oral contraceptives or spironolactone may be
useful for female-pattern hair loss associated with hyperandrogenemia,...

Based on the provided information, I would rate the relevance of the response as 4 out of 5. The
response directly addresses the user's question by providing a list of effective treatments for
sudden patchy hair loss, including topical and oral corticosteroids, immunomodulators, minoxidil,
anthralin, and phototherapy. The response also cov

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Factual claim 1 (topical
corticosteroids): Supported by the context. * Factual claim 2 (intralesional corticosteroids):
Supported by the context. * Factual claim 3 (immunomodulators): Supported by the context. * Factual
claim 4 (oral antibiotics): Not explicitly stated in the context, so this claim is not grounded.
Overall, I would rate the answer as "Mostly grounded" (rating 4) because most of the factual claims
are supported by the context, but one claim (oral antibiotics) is not explicitly stated.

Based on the provided context, I would rate the relevance of the answer as 4 out of 5. The response
directly addresses the user's question by providing a list of effective treatments for sudden patchy
hair loss, including topical and systemic corticosteroids, immunomodulators, and hormonal
modulators. The answer also covers key requested details such as the causes of alopecia areata and
the different types of hair loss.

...

Based on the provided context, I would rate the answer as follows:  ###Rating(1-5): 4 - Mostly
grounded  The answer provides several effective treatments and solutions for sudden patchy hair
loss, including topical corticosteroids, oral antimalarials, immunotherapy, and wigs or hairpieces.
These treatments are supported by the context, which mentions the use of these treatments for
various types of hair loss. However, there is some unsupported information in the answer, such as
the claim that alopecia areata is an autoimmune disorder affecting genetically susceptible people
exposed to unclear environmental triggers. This statement is not ex...

Based on the provided information, I would rate the relevance of the answer as a 4 out of 5. The
response directly addresses the user's question by providing effective treatments and possible
causes of sudden patchy hair loss, which is the main focus of the query. The answer also covers most
key requested details, such as topical corticosteroids

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context and answer, I would rate the groundedness of the response as follows:
Rating: 4 (Mostly grounded)  The answer provides several effective treatments and solutions for
addressing sudden patchy hair loss, which are supported by the retrieved context. The context
provides information on the etiology of alopecia, including androgenetic alopecia, drug-induced
alopecia, and infection. The answer correctly references the use of medications such as minoxidil
(Rogaine) and finasteride (Propecia) to treat androgenetic alopecia, which is a common cause of
patchy hair loss. Additionally, the answer mentions low-level laser...

Based on the provided information, I would rate the relevance of the answer as a 4 out of 5. The
answer directly addresses the user's question by providing effective treatments and solutions for
sudden patchy hair loss, which is a common cause of alopecia. The response also covers most key
aspects of the user's request, including the possible cau

Llama.generate: prefix-match hit


Error generating groundedness response 1: Requested tokens (2458) exceed context window of 2300
Error generating relevance response 2: Requested tokens (2419) exceed context window of 2300
Error generating groundedness response 1: Requested tokens (2458) exceed context window of 2300

Error generating relevance response 2: Requested tokens (2419) exceed context window of 2300


### ✅Best Scores: **#2** == **#3** == **#4**

**#2 Custom Parameters:** k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1

**#3 Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

**#4 Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### 📊Scoring Summaries:

**Scoring for #2**
* Groundedness (Rating 4 — Mostly grounded): The response is mostly grounded (4/5) because the main treatments listed (topical corticosteroids, intralesional corticosteroids, and immunomodulators) are supported by the provided context. However, the mention of oral antibiotics is not explicitly supported, which introduces a small ungrounded claim.

* Relevance (Rating 4 — High relevance): The response is highly relevant (4/5) since it directly answers the question by listing treatment options and discussing possible causes behind sudden patchy hair loss. It loses a point because it doesn’t give enough practical patient-facing detail, like expected effectiveness, side effects, and how treatment choice depends on the cause.

**Scoring for #3**
* Groundedness (Rating 4 — Mostly grounded): The groundedness is 4/5 (mostly grounded) because most of the treatments listed (topical corticosteroids, oral antimalarials, immunotherapy, wigs/hairpieces) are supported by the provided context. It loses some groundedness due to an added claim about alopecia areata’s autoimmune mechanism and genetic/environmental triggers that isn’t explicitly stated in the context.

* Relevance (Rating 4 — High relevance): The relevance is 4/5 because the response clearly answers what the user asked by giving treatment options and touching on possible causes. It’s slightly less relevant because it could better cover the full range of causes (like differentiating alopecia areata vs. telogen effluvium or infections) instead of mainly focusing on treatments.

**Scoring for #4**
* Groundedness (Rating 4 — Mostly grounded): The groundedness is 4/5 (mostly grounded) because the response’s main treatment claims (ex: minoxidil, finasteride, and low-level laser therapy) align with what the provided context supports for common alopecia-related causes. It’s not a perfect 5 because some treatment recommendations may be slightly overgeneralized for “patchy bald spots” (which often suggests alopecia areata) without fully tying each option directly back to the specific subtype described in the context.

* Relevance (Rating 4 — High relevance): The relevance is 4/5 because the answer addresses the user’s exact intent by listing treatments and giving possible causes of sudden patchy hair loss. It falls short of a 5 since it doesn’t clearly break down the most likely causes of true patchy/scalp-spot hair loss (like alopecia areata vs fungal infection) or explain when each treatment is appropriate, which matters for patient usefulness.
---
### ⚠️ Other Scores:

* **#1:** Groundedness score of 1. Revelance score of 4.

### 🎯 Partial Scoring Response(s): None

### 🚫 Invalid Response(s): **#5**

## Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [26]:
varied_func_and_params(generate_ground_relevance_response, user_query4, FLAG)

1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Supportive care: Fully
grounded (5) - The context provides detailed information on supportive care, including preventing
systemic complications, providing good nutrition, and preventing pressure ulcers. * Rehabilitation
therapy: Moderately grounded (3) - The context mentions early intervention by rehabilitation
specialists and the importance of cognitive therapy, but does not provide specific details on the
types of therapy or their duration. * Cognitive therapy: Fully grounded (5) - The context provides
extensive information on cognitive therapy, including its importance...

Based on the provided information, I would rate the relevance of the answer as 4 out of 5. The
response provides a comprehensive list of treatments for a person who has sustained a physical
injury to brain tissue, resulting in temporary or permanent impairment of brain function. The
treatments mentioned are relevant to the question and cover mos

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Supportive care: Fully
grounded (5/5) - The context provides detailed information about the importance of supportive care
in preventing systemic complications and ensuring good nutrition and hygiene. * Rehabilitation
therapy: Moderately grounded (4/5) - While the context mentions the importance of rehabilitation
therapy, it does not provide specific information about the types of therapy or their duration. *
Cognitive therapy: Weakly grounded (2/5) - The context briefly mentions cognitive therapy but does
not provide any specific information about its use or effectiveness...

Based on the provided context and answer, I would rate the relevance as follows:  ###Answer: 4 -
High relevance.  The answer addresses most key aspects of the question, providing information on
treatments for a person who has sustained a physical injury to brain tissue resulting in temporary
or permanent impairment of brain function. The respons

...

Based on the provided context and answer, I would rate the groundedness of the response as follows:
Rating: 4 (Mostly grounded)  The main points of the answer are supported by the context, but there
are some minor unsupported details. Here's a breakdown of the rating:  1. Early intervention by
rehabilitation specialists is recommended to prevent secondary disabilities and maximize functional
recovery. (Supported by the context) 2. Cognitive therapy is recommended for patients with severe
cognitive dysfunction. (Supported by the context) 3. Surgical or nonsurgical immobilization of the
affected area is recommended to prevent complications s...

Based on the provided information, I would rate the relevance of the answer as a 4 out of 5. The
answer directly addresses the question by providing recommended treatments for a person who has
sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain
function. The answer covers most key aspects of the re

Llama.generate: prefix-match hit


Error generating groundedness response 1: Requested tokens (2320) exceed context window of 2300


Llama.generate: prefix-match hit


Error generating groundedness response 1: Requested tokens (2320) exceed context window of 2300

Based on the provided information, I would rate the relevance of the given answer as follows


### ✅Best Scores: **#3** == **#4**

**#3 Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

**#4 Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### 📊Scoring Summaries:

**Scoring for #3**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because most of the listed treatments (ex: supportive care like preventing complications, nutrition, and pressure-ulcer prevention) match what the provided context supports. It loses points because it includes at least one extra detail not clearly supported by the context, such as the specific timeline claim about recovery from a vegetative state.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 because the answer generally fits the user’s question by providing a treatment-focused overview for brain tissue injury and impairment. It’s not a perfect 5 since it doesn’t fully cover key care elements the question implies (ex: complication management, secondary disability prevention, and realistic recovery considerations) in a clear, structured way.

**Scoring for #4**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because the core recommendations (early rehab involvement, cognitive therapy, and general complication prevention) are supported by the provided context. It’s not a perfect 5 since a few details (like specific immobilization or added care elements) appear to go slightly beyond what is explicitly stated in the retrieved text.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 because the answer clearly focuses on recommended treatments for brain tissue injury, matching the user’s question. It loses points because it doesn’t fully cover the full range of expected care steps (ex: acute stabilization, monitoring for complications like seizures/ICP, and longer-term follow-up planning), so the response feels somewhat incomplete even though it’s on-topic.

---
### ⚠️ Other Scores: None

### 🎯 Partial Scoring Response(s):

* **#1:** Revelance score of 4. No groundedness score.
* **#2:** Revelance score of 4. No groundedness score.

### 🚫 Invalid Response(s): **#5**

## Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [27]:
varied_func_and_params(generate_ground_relevance_response, user_query5, FLAG)

1. Default parameters: k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Factual claims that are
explicitly stated in the context: 4/5     + The necessary precautions and treatment steps for a
person who has fractured their leg during a hiking trip are immobilization, rest, ice, compression,
and elevation (RICE), pain management, reduction, surgical intervention, and monitoring for
complications.    + The context provides specific information about the types of injuries that may
occur during a hiking trip and the appropriate treatment methods. * Factual claims that are not
explicitly stated in the context but can be inferred: 1/5      + There is no men...

Based on the provided information, I would rate the relevance of the response as a 4 out of 5. The
response directly addresses the user's question by providing a comprehensive list of necessary
precautions and treatment steps for a person who has fractured their leg during a hiking trip. The
response covers all the key requested details

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided context, I would rate the answer as follows:  * Factual claims that are
explicitly stated: 4/5    + The leg should be immobilized using a splint or cast to prevent further
injury and promote healing.   + Pain should be managed with opioids or other analgesics as needed.
+ Rest, ice, compression, and elevation (RICE) are recommended for treatment. * Factual claims that
are clearly supported by the context: 4/5   + The use of RICE for soft-tissue injuries with or
without musculoskeletal injuries is supported by the provided context.       + The need for
definitive treatment, such as reduction, for certain injuries is supported b...

Based on the provided information, I would rate the relevance of the response as a 4 out of 5. The
response provides a comprehensive overview of the necessary precautions and treatment steps for a
person who has fractured their leg during a hiking trip, including immobilization, pain management,
and preventive measures against deep vein 

...

Based on the provided context, I would rate the answer as follows:  ###Rating(1-5): 4 - Mostly
grounded  The answer provides a list of necessary precautions and treatment steps for a person who
has fractured their leg during a hiking trip. The information is supported by the context, which
includes a medical manual and a specific chapter on exercise and sports injury. The answer is mostly
grounded, with some minor unsupported details (eg, the specific type of immobilization device used).
However, there are no hallucinations or contradictions in the answer.  Here's a breakdown of the
rating:  1. Immobilization: The context supports the use o...

Based on the provided information, I would rate the relevance of the response as a 4 out of 5. The
response directly addresses the user's question by providing necessary precautions and treatment
steps for a person who has fractured their leg during a hiking trip. The response covers most key
requested details, such as immobilization, pain manag

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


Based on the provided information, I would rate the groundedness of the answer as follows:  Rating:
4 (Mostly grounded)  The answer provides a clear and accurate summary of the necessary precautions
and treatment steps for a person who has fractured their leg during a hiking trip. The information
is supported by the provided context, which includes a reference to the Merck Manual of Medical
Diagnosis and Therapy. However, there are some minor unsupported details, such as the specific
exercises that should be done under the supervision of a healthcare professional. Overall, the
answer is well-grounded and provides reliable information for th...

Based on the information provided, I would rate the relevance of the answer as a 4 out of 5. The
answer provides a good overview of the necessary precautions and treatment steps for a person who
has fractured their leg during a hiking trip, including rest, ice, compression, stretching, and
strengthening exercises. However, the answer is slightly

Llama.generate: prefix-match hit


Error generating groundedness response 1: Requested tokens (2458) exceed context window of 2300
Error generating relevance response 2: Requested tokens (2419) exceed context window of 2300
Error generating groundedness response 1: Requested tokens (2458) exceed context window of 2300

Error generating relevance response 2: Requested tokens (2419) exceed context window of 2300


### ✅Best Scores: **#1** == **#2** == **#3** == **#4**

**#1 Default Parameters:** k_docs=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50

**#2 Custom Parameters:** k_docs=3, max_tokens=175, temperature=0.4, top_p=0.5, top_k=1

**#3 Custom Parameters:** k_docs=2, max_tokens=250, temperature=0.7, top_p=0.0, top_k=100

**#4 Custom Parameters:** k_docs=1, max_tokens=200, temperature=0.1, top_p=0.3, top_k=25

### 📊Scoring Summaries:

**Scoring for #1**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because the key care steps listed (immobilization, RICE, pain control, reduction/surgery when needed, and monitoring for complications) are supported by the provided context. It isn’t a perfect 5 because some parts are more general or inferred rather than clearly tied to exact wording/details in the context.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 since the response directly answers the question by outlining the main precautions and treatment steps for a hiking-related leg fracture, including immediate care and follow-up considerations. It loses a point because it’s a bit light on actionable detail, like specific red flags (loss of pulse/sensation, open fracture) and practical instructions for safe transport and recovery planning.

**Scoring for #2**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because the core recommendations—immobilization (splint/cast), RICE, and pain control—are directly supported by the provided context. It’s not a perfect 5 because some details (like opioids specifically or certain add-ons) may be more generalized or not explicitly stated word-for-word in the context.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 since the response clearly covers the immediate management steps for a leg fracture during a hike (stabilization, pain management, swelling control, complication prevention). It loses a point because it doesn’t fully address recovery planning, like follow-up care timelines, mobility/rehab considerations, or what to monitor during healing.

**Scoring for #3**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because the response’s main steps (like immobilization, splinting, pain management, and definitive treatment) align well with the provided medical context. It’s not a perfect 5 since some specifics (like exact immobilization device choices) may be slightly assumed rather than explicitly stated in the retrieved text.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 because the response clearly answers what to do for a fractured leg during a hike, covering the major emergency and treatment actions. It loses a point because it doesn’t fully cover recovery-focused considerations, like long-term aftercare, red-flag complications to watch for, or detailed guidance on rehab and follow-up.

**Scoring for #4**
* Groundedness (Rating 4 — Mostly grounded): The groundedness score is 4/5 (mostly grounded) because most of the advice (immobilization/rest, icing, compression, and recovery steps) matches what would typically be supported by an authoritative clinical reference like the Merck Manual. It’s not a perfect 5 because it introduces minor details not clearly backed by the context, such as specific exercise recommendations without explicit supporting text.

* Relevance (Rating 4 — High relevance): The relevance score is 4/5 because it mostly answers the question by outlining immediate precautions and general care/recovery considerations for a hiking-related leg fracture. It loses points because it misses or glosses over key specifics the user asked for (like exact exercise types and immobilization duration) and includes unrelated filler content (like an email address or sharing warning) that doesn’t help the patient.

---
### ⚠️ Other Scores: None

### 🎯 Partial Scoring Response(s): None

### 🚫 Invalid Response(s): **#5**

## Overall Scoring Observations

---
**Results:**

Best Scoring List: 1, 2, 2, 3, 3, 3, 3, **4, 4, 4, 4, 4**

Invalid Scoring Responses: 1, **5, 5, 5, 5, 5**

---
**Conclusions:**
The best parameter combination that scored in the best scores section over all 5 queries is custom parameter combination #4. The custom parameter combination #5 threw an error for every query because it was exceeding the context window limit of 2300. This was probably because it was using k_docs = 4 combined with both max_tokens = 300 and temperature = 1.0. These 3 parameter values are the largest values used in the entire project so it makes sense that it is exceeding the context window limit. After analyzing the non-best and non-invalid scores, I have determined that default parameter combination #1 has the worst overall scoring average.

Best Grounding and Revelance Scoring: **#4**

Worst Grounding and Revelance Scoring: **#1**

Greatest Number of Invalid Scoring Responses: **#5**



# Actionable Insights and Business Recommendations

## Actionable Insights:

**Prompt performance pattern (why and impact):**

* Prompt #3 is best in 2/5 prompt comparisons while prompt #1 is worst in 3/5, indicating that prompts emphasizing evidence-based tone with transparent limitations generate higher-quality responses. We should align the prompt wording with evidence-based guidance to avoid overly rigid refusal wording that truncates answers.

**Parameter stability vs quality (why and impact):**

* Parameter combo #4 appears most often in the best groundedness/relevance scores, while combo #1 has the worst overall average. I think favoring conservative sampling and moderate output length will reduce hallucinations that cut off important details could be crucial to creating a thorough response.

**Context window overload (why and impact):**

* Combo #5 fails across queries due to the 2300-token context window being exceeded (k_docs=4, max_tokens=300, temperature=1.0). We should create a watcher function that keeps the combined retrieval and generation data within the available window to avoid invalid responses.

**Answer completeness gaps (why and impact):**

* Evaluator function documentation shows that groundedness/relevance scores often plateau at 4/5 due to missing red flags and recovery specifics. We should require answers to include safety-critical details so responses move from "mostly grounded"(4) to "fully grounded"(5).

**Safety risk from early‑activity advice (why and action):**

* The worst-rated answers & responses include premature stretching/strengthening after fractures or splints, which was flagged as unsafe. We should add a safety checks that blocks rehab advice unless stabilization/clinician clearance is stated. In fact, there is a lot of dangerous advice that the LLM could provide that could hurt the patient. A wide range of safety checks should be put in place to protect patients from harmful or misleading advice.

**Response length trade‑off (why and action):**

* Longer outputs (e.g., response #5) are often judged more useful, but can become invalid when token limits are exceeded. We should target a bounded length range with stricter retrieval to preserve completeness without overflow to avoid the invalid responses that were retrieved from response #5.

## Business Recommendations:

**Context-budget guardrails:**

* Implement a token budgeter that trims retrieved chunks or lowers max_tokens before generation when the context approaches 2300 tokens, and log every adjustment for auditability.

**Safety and response structure:**

* Update the answer template to include required sections for urgent red flags and follow-up care, and add a validation check that flags outputs missing these sections during evaluation.

**Evaluation and monitoring cadence:**

* Automate weekly groundedness/relevance scoring on the five benchmark queries and add a human safety review sample (e.g., 10 responses/week) to catch unsafe or incomplete guidance.

**Production defaults and rollout:**

* Ship to production with Prompt #3 and parameter combo #5 as the default configuration, and reserve Prompt #2 and parameter combo #5 as well as Prompt #3 and parameter combo #4 as a controlled fallbacks for split testing and regression checks.

**Audit and compliance logging:**

* Store retrieved sources, prompt versions, and parameter combination settings with each response to support code review and security audits.

<font size=6 color='blue'>Power Ahead</font>
___